# RoPE attention is an exact forward-pass gradient step with softmax intact
**Executable companion: exact effective weights, positional derivatives, and pretrained Qwen measurements.**

Open this notebook in Colab, select a **GPU runtime**, then choose **Run all**. No original scripts or cached-activation uploads are needed. The final cell downloads a ZIP containing the complete measurements, exact replay factors, executable source, environment information, and checksums.

The default `paper` profile measures **Qwen2.5-0.5B, layer 23, 16 distinct article prefixes, 32 continuations per prefix, 512 tokens, seed 0**. `quick` exercises the same pretrained pipeline on a smaller sample; `smoke` tests a tiny random Qwen without downloading a checkpoint. These profiles are labeled in every release bundle.

Every query uses its own exact matrix. The frozen-anchor experiment and the linearized-softmax baseline are separately named controls. The native model retains full softmax and all Q/K/V projection biases.


In [ ]:
#@title 1. Select the run
PROFILE = "paper" #@param ["paper", "quick", "smoke"]
AUTO_DOWNLOAD = True #@param {type:"boolean"}

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, subprocess, sys, tempfile, time, zipfile

assert PROFILE in {"paper", "quick", "smoke"}
MODEL = "Qwen/Qwen2.5-0.5B"
MODEL_REVISION = "060db6499f32faf8b98477b0a26969ef7d8b9987"
DATASET_REVISION = "b08601e04326c79dfdd32d625aee71d232d685c3"
CONFIG = {
    "paper": dict(contexts=16, queries=32, length=512, layer=23, batch=2, seed=0),
    "quick": dict(contexts=1, queries=4, length=128, layer=23, batch=2, seed=0),
    "smoke": dict(contexts=1, queries=4, length=24, layer=-1, batch=2, seed=0),
}[PROFILE]
BASE_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
RUN_ROOT = Path(tempfile.mkdtemp(prefix=f'rope_softmax_{PROFILE}_', dir=BASE_DIR))
SOURCE_DIR = RUN_ROOT / 'source'
LOG_DIR = RUN_ROOT / 'logs'
SOURCE_DIR.mkdir(); LOG_DIR.mkdir()
RESULTS = RUN_ROOT / 'recordings'
ENV_DIR = BASE_DIR / 'rope_softmax_environment'
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0',
               OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
               HF_HUB_DISABLE_PROGRESS_BARS='1', HF_HUB_DISABLE_TELEMETRY='1',
               HF_HOME=str(BASE_DIR / 'rope_softmax_download_cache'))
RUN_MANIFEST = dict(notebook_protocol=1, profile=PROFILE, config=CONFIG, model=MODEL,
    model_revision=MODEL_REVISION, dataset='Salesforce/wikitext',
    dataset_config='wikitext-2-raw-v1', dataset_split='test', dataset_revision=DATASET_REVISION,
    started_utc=datetime.now(timezone.utc).isoformat(), status='configured', commands=[])

def save_manifest():
    (RUN_ROOT / 'run_manifest.json').write_text(json.dumps(RUN_MANIFEST, indent=2, allow_nan=False)+'\n')

def run_command(command, name, quiet=False):
    command = [str(x) for x in command]
    started = time.monotonic()
    item = dict(name=name, command=command)
    RUN_MANIFEST['commands'].append(item); save_manifest()
    lines = []
    with (LOG_DIR / f'{name}.log').open('w') as log:
        with subprocess.Popen(command, cwd=SOURCE_DIR, env=RUN_ENV, text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1) as proc:
            for line in proc.stdout:
                log.write(line); log.flush()
                lines.append(line)
                if not quiet:
                    print(line, end='', flush=True)
            code = proc.wait()
    item.update(exit_code=code, seconds=round(time.monotonic()-started, 3))
    if code:
        RUN_MANIFEST['status'] = 'failed'; save_manifest()
        if quiet:
            print(''.join(lines[-50:]))
        raise RuntimeError(f'{name} failed (exit {code}). Full log: {LOG_DIR / (name + ".log")}')
    save_manifest()
    if quiet:
        print(f'{name}: passed ({item["seconds"]:.1f} s)')
    return ''.join(lines)

save_manifest()
print(json.dumps(dict(profile=PROFILE, **CONFIG), indent=2))
print('Results:', RUN_ROOT)


## Exact construction and the omitted term
Let $u_i=[x_i,1]$ be the normalized attention input with an augmented bias coordinate, $K$ the cached rotated keys, and $V$ the values. At a fixed query position, put $L=W_q^{\mathrm{aug}}R(p_i)^\top/\sqrt d$, $a_j=L\tilde k_j^\top$, $s_{ij}=u_i a_j$, $Z_i=\sum_j e^{s_{ij}}$, and $\mu=N^{-1}\sum_j v_j$. Then
$$
\rho(s)=\begin{cases}(e^s-1)/s&s\ne0,\\1&s=0,\end{cases}\quad
c_{ij}=\rho(s_{ij})/Z_i,\quad
\Delta M_i=\sum_j c_{ij}a_j(v_j-\mu)=L T_i,\qquad y_i=\mu+u_i\Delta M_i.
$$
The proof is $y_i-\mu=Z_i^{-1}\sum_j(e^{s_{ij}}-1)(v_j-\mu)=\sum_j c_{ij}s_{ij}(v_j-\mu)$. The implementation evaluates this identity at the **original scores** using rescaled exponentials and a removable-singularity calculation; it never changes the declared matrix gauge.

For each query, define $\mathcal E_i(B)=\tfrac12\sum_j c_{ij}\|a_j^\top B-(v_j-\mu)\|^2$, holding its coefficients, features, and targets fixed when differentiating with respect to $B$. A unit step from zero gives $B^+=-\nabla_B\mathcal E_i(0)=\Delta M_i$. The notebook checks this equality with PyTorch autograd as well as checking the softmax readout independently.

Freezing anchor $a$ omits **exactly**
$$
y_i-(\mu+u_i\Delta M_a)=u_i(\Delta M_i-\Delta M_a).
$$
Adding this term restores the full query-dependent output. The residual write is $h_i\mapsto h_i+(\mu+u_i\Delta M_i)W_o$; $u_i$ is formed **after RMSNorm**. The matrices are secant representations; the positional derivative is separately checked through $\partial_p R(p)x=A R(p)x$ and the differentiated softmax, with content and cached keys/values held fixed.


## Embedded implementation
The measurement engine below is the unchanged script identified by the supplied pretrained run's SHA-256. This notebook calls its `--shared-update` and exact replay paths. The two additional modules supply every-anchor controls and independent release checks. All source is included here and exported with the results.


In [ ]:
#@title 2a. Embedded measurement engine (expand to inspect)
_ = (SOURCE_DIR / 'frozen_rope_gradient_test.py').write_text(r'''"""Frozen RoPE intervention pilot; corrected mathematical and evaluation protocol.

Usage (edit defaults below or pass arguments):
  python -m pip install torch 'transformers==4.51.3' datasets numpy
  python frozen_rope_gradient_test.py --self-test   # NumPy only, no downloads
  python frozen_rope_gradient_test.py --smoke      # tiny random model, CPU
  python frozen_rope_gradient_test.py --layer -1 --gate none --out results_v2
  python frozen_rope_gradient_test.py --gate chi   # optional strict hypothesis
  python frozen_rope_gradient_test.py --scan-only --out qwen_screen
  python frozen_rope_gradient_test.py --screen-and-test --out rope_screen
  python frozen_rope_gradient_test.py --shared-update --length 256 --contexts 16 --queries 32 --out qwen_shared
  python frozen_rope_gradient_test.py --shared-update --smoke --out shared_smoke
  python frozen_rope_gradient_test.py --replay qwen_shared --out qwen_exact

Version 5 makes the two different questions explicit. The literal identity
uses each query's OWN DeltaM_i and is checked against direct softmax. Freezing
DeltaM_0 is a separate approximation; poor transfer cannot falsify the identity.
--replay reads the saved weights.npz and context_*.npz from version 4 or later,
without torch, downloads, or a new model forward pass. It verifies the exact
query-dependent correction to the frozen readout and the literal FULL-attention
matrix, including each query's self key/value and its query-dependent value mean.
Fresh --shared-update runs also generate this exact replay report automatically.
Original frozen controls are retained, with no fitted gate or removed heads.

Version 4 adds --shared-update, a separate Section 7A measurement, with no
training, gradients, eligibility gate, or fitted comparator. It ports the
literal effective() factorization, rho(s)=(exp(s)-1)/s at ORIGINAL scores.
Natural one-token continuations of a common prefix supply different normalized
query activations at one fixed position. Their self K/V entries are EXCLUDED
from this conditional readout so every query attends the identical prefix.
The unchanged native attention, including self, is verified separately.
Qwen's projection biases use homogeneous coordinates [normalized_hidden, 1].
This is a matrix on attention INPUTS, not on the pre-RMSNorm residual stream.
All heads and the sum of projected head updates are measured. Outputs include
diagnostics.csv, heads.csv, per_query.csv, REPORT.md, metadata.json, and exact
matrix factors in context_*.npz plus weights.npz; --save-matrices also writes
dense per-head DeltaM. This stage does not run the older intervention pilot.
A tiny random Qwen hook/round-trip preflight runs automatically before a real
checkpoint is downloaded; its output is retained in metadata.json. --smoke
runs that check alone and labels its results as random, not pretrained.

Version 3 screening protocol: --scan-only records every layer in ONE forward
pass per calibration block, without gradients, interventions, or held-out data.
It lists heads with chi >= .5, mean lag in [1,30], and valid fraction >= .25.
--screen-and-test first screens Qwen, then reruns each qualifying layer with
the strict gate and UNCAPPED corrections; if Qwen has no qualifying heads, it
screens meta-llama/Llama-3.2-1B once and follows the same rule. Llama downloads
require the user's existing HF access; access errors are not negative results.
All tested layers are reported, with a shared multiple-comparison family.
The uncapped step is an experimental extrapolation, NOT a certified loss step.
No eligible heads means no support for THESE criteria on THESE sampled blocks,
not proof that the checkpoint lacks all useful positional mechanisms.

Only ONE selected layer is intervened on per run. All controls therefore see
identical incoming activations. Baseline is the native, unmodified model.
The default tests all numerically usable heads; chi is a diagnostic, not a
required threshold. Use --gate chi to test the separate high-correlation
hypothesis. An empty gate writes calibration diagnostics and stops BEFORE
held-out evaluation. No automatic threshold relaxation is performed.
Version 2 changes the experimental protocol after inspecting calibration
statistics from the initial empty-gate run. Results are exploratory. Reusing
already inspected held-out articles does not provide independent confirmation.
Use a distinct output directory for each run.

Mathematics, with content and attended keys held fixed:
  a_j = (A q_rot).k_rot_j / sqrt(d);  y' = Cov_w(v,a).
  x_j = j-p; gamma = Cov_w(a,x); chi = gamma^2/(Var_w(a) Var_w(x)).
  projected_slope = y' * gamma/(Var_w(a) Var_w(x)).
This is the component of the weighted value-position regression slope captured
by a. It equals the full regression slope when chi=1; chi alone does NOT make
y' a unit-gain value-position slope. No process-derivative claim follows.

Projected/regression corrections use (mean_lag + lookahead)*slope, followed by
a per-row relative norm cap. Lookahead is a hypothesis, not implied by shifted
next-token labels. Phase mode instead uses phase_step*y', WITHOUT mean lag.
Alpha=1 means the capped policy, not a guaranteed optimum or a Taylor guarantee.

Changes: signed/gain-corrected projection; exact native baseline; one-layer
isolation; centered moments; no forced gate; bounded updates; independent
finite-difference checks; genuine norm-matched permutation/random controls;
signed calibration grid; held-out increment errors; paired article bootstrap
contrasts with multiplicity adjustment; explicit inconclusive verdicts.

Limitations: full, fixed-frequency RoPE (default or llama3); eager full causal
attention; no cache/padding;
fp32 experiment only. Gradient calibration needs backward passes but does not
update model weights. This is a pilot, not proof of sequence derivatives,
loss-surface smoothing, inference optimization guarantees, or novelty.
"""

import argparse
import csv
import hashlib
import importlib.metadata
import json
import math
from pathlib import Path
import re
import subprocess
import sys
import tempfile

import numpy as np

MODEL_ID = "Qwen/Qwen2.5-0.5B"
SEQ_LEN, BATCH, N_CAL_BLOCKS, N_TEST_BLOCKS = 512, 2, 16, 32
LOOKAHEAD, SLOPE = 0.0, "projected"
CHI_GATE, MU_GATE = 0.5, 1.0
MAX_REL_UPDATE, PHASE_STEP = 0.02, 0.1
SEED, N_BOOT, NULL_SEEDS = 0, 4000, 3


def write_csv(path, rows, fieldnames=None):
    if rows or fieldnames:
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames or list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


def section7a_coefficients(scores):
    """Literal rho(s)/sum(exp(s)); an evaluation scale never changes rho's s.

    Ported from the validated lol.py effective() implementation. The degree-13
    series handles the removable singularity at zero. Some very negative raw
    scores make this particular matrix numerically unrepresentable even when
    softmax itself is well conditioned; those cases fail explicitly.
    """
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim < 1 or scores.shape[-1] == 0 or not np.isfinite(scores).all():
        raise ValueError("Section 7A needs a nonempty set of finite original scores.")
    shift = np.maximum(scores.max(axis=-1, keepdims=True), 0.)
    small = np.abs(scores) < .1
    safe = np.where(small, 1., scores)
    es, e0 = np.exp(scores-shift), np.exp(-shift)
    z = np.where(small, scores, 0.)
    polynomial = np.full_like(z, 1/math.factorial(14))
    for power in range(12, -1, -1):
        polynomial = polynomial*z+1/math.factorial(power+1)
    denominator = es.sum(axis=-1, keepdims=True)
    if np.any(denominator == 0):
        raise FloatingPointError("Literal original-score coefficients exceed float64 range; no gauge change was made.")
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        result = np.where(small, e0*polynomial, (es-e0)/safe)/denominator
    if not np.isfinite(result).all():
        raise FloatingPointError("Nonfinite literal Section 7A coefficients; no gauge change was made.")
    return result


def section7a_rotate(rows, cosine, sine):
    """HF/Qwen split-half RoPE, on row vectors; NOT adjacent-pair RoPE."""
    d = rows.shape[-1]
    if d % 2 or cosine.shape[-1] != d or sine.shape[-1] != d:
        raise ValueError("Invalid full-RoPE dimensions.")
    half = np.concatenate((-rows[..., d//2:], rows[..., :d//2]), axis=-1)
    return rows*cosine+half*sine


def section7a_effective(query, wq, keys, values, cosine, sine, scale):
    """Exact affine Section 7A, one head and a FIXED attended bank/position.

    query: [Q,m+1], wq: [m+1,d], keys: [N,d] already rotated,
    values: [N,r]. Last query coordinate is 1; last wq row is q_proj.bias.
    L = Wq_aug R_query^T * scale; T_i = K_rot^T diag(rho(s_i)/Z_i)(V-mu).
    Returns literal DeltaM_i = L @ T_i, not a Jacobian or a fitted matrix.
    """
    query, wq, keys, values, cosine, sine = [np.asarray(x, dtype=np.float64)
        for x in (query, wq, keys, values, cosine, sine)]
    if (query.ndim != 2 or wq.ndim != 2 or keys.ndim != 2 or values.ndim != 2
            or query.shape[1] != wq.shape[0] or wq.shape[1] != keys.shape[1]
            or len(keys) != len(values) or not len(keys)
            or cosine.ndim != 1 or sine.ndim != 1):
        raise ValueError("Incompatible fixed-bank Section 7A shapes.")
    if not all(np.isfinite(x).all() for x in (query, wq, keys, values, cosine, sine)):
        raise FloatingPointError("Nonfinite Section 7A inputs.")
    left = section7a_rotate(wq, cosine, sine)*scale
    scores = (query@left)@keys.T
    coef = section7a_coefficients(scores)
    probability = np.exp(scores-scores.max(-1, keepdims=True))
    probability /= probability.sum(-1, keepdims=True)
    mu = values.mean(0)
    weighted = coef[..., None]*(values-mu)[None]
    fast = keys.T[None]@weighted
    matrix = left[None]@fast
    # Independent multiplication order explicitly sums the Section 7A slopes.
    direct = (left@keys.T)[None]@weighted
    native = probability@values
    rebuilt = mu+np.einsum("qm,qmr->qr", query, matrix)
    np.testing.assert_allclose(matrix, direct, atol=2e-9, rtol=2e-8,
                               err_msg="Literal slope sum differs from effective() matrix.")
    np.testing.assert_allclose(rebuilt, native, atol=2e-9, rtol=2e-8,
                               err_msg="Literal Section 7A reconstruction failed; do not interpret its statistics.")
    return dict(left=left, fast=fast, matrix=matrix, mu=mu, native=native,
                rebuilt=rebuilt, scores=scores, probability=probability,
                matrix_error=float(np.max(np.abs(matrix-direct))),
                output_error=float(np.max(np.abs(rebuilt-native))))


def section7a_energy(matrices, output_projection=None):
    """Per-query ||M Wo||_F^2 without materializing m-by-model_width updates."""
    matrices = np.asarray(matrices, dtype=np.float64)
    if output_projection is None:
        return np.einsum("qmr,qmr->q", matrices, matrices)
    gram = output_projection@output_projection.T
    answer = np.einsum("qmr,rs,qms->q", matrices, gram, matrices, optimize=True)
    # Only final rounding of a PSD quadratic form is clipped, not the matrices.
    return np.maximum(answer, 0.)


def section7a_ratio(numerator, denominator, root=False):
    if denominator <= 0 or not np.isfinite(numerator+denominator):
        return None
    result = max(float(numerator/denominator), 0.)
    return math.sqrt(result) if root else result


def section7a_transfer(native, frozen, mu):
    """Query 0 supplies the frozen matrix; all transfer errors exclude query 0."""
    if len(native) < 2:
        raise ValueError("Frozen transfer requires at least two queries.")
    error = float(np.mean((frozen[1:]-native[1:])**2))
    constant = float(np.mean((native[:1]-native[1:])**2))
    mean_value = float(np.mean((np.asarray(mu)-native[1:])**2))
    ratio = section7a_ratio(error, constant)
    return dict(frozen_update_transfer_mse=error,
                constant_output_baseline_mse=constant,
                mean_value_baseline_mse=mean_value,
                frozen_transfer_relative_rmse=section7a_ratio(error, constant, root=True),
                frozen_transfer_r2_vs_constant=None if ratio is None else 1-ratio)


def section7a_head_statistics(result, query, wo):
    matrix = result["matrix"]
    centered = matrix-matrix.mean(0, keepdims=True)
    energy = float(section7a_energy(matrix, wo).sum())
    variation = float(section7a_energy(centered, wo).sum())
    raw_energy = float(section7a_energy(matrix).sum())
    raw_variation = float(section7a_energy(centered).sum())
    linear_energy = float(section7a_energy(matrix[:, :-1], wo).sum())
    linear_variation = float(section7a_energy(centered[:, :-1], wo).sum())
    frozen_head = result["mu"]+query@matrix[0]
    native, frozen, mu = result["native"]@wo, frozen_head@wo, result["mu"]@wo
    stats = dict(projected_matrix_energy=energy, projected_matrix_variation_energy=variation,
        raw_matrix_energy=raw_energy, raw_matrix_variation_energy=raw_variation,
        linear_matrix_energy=linear_energy, linear_matrix_variation_energy=linear_variation,
        literal_update_query_variation=section7a_ratio(variation, energy, root=True),
        raw_matrix_query_variation=section7a_ratio(raw_variation, raw_energy, root=True),
        linear_update_query_variation=section7a_ratio(linear_variation, linear_energy, root=True),
        matrix_max_abs_error=result["matrix_error"], output_max_abs_error=result["output_error"],
        original_score_min=float(result["scores"].min()), original_score_max=float(result["scores"].max()),
        frozen_update_transfer_head_mse=float(np.mean((frozen_head[1:]-result["native"][1:])**2)),
        **section7a_transfer(native, frozen, mu))
    return stats, native, frozen, mu


def section7a_summarize(rows):
    """Pool WITHIN-context matrix variation; contexts are the replicate unit."""
    summary = []
    for head in sorted({r["head"] for r in rows}, key=lambda x: (x == "all", 0 if x == "all" else int(x))):
        group = [r for r in rows if r["head"] == head]
        good = [r for r in group if r["status"] == "passed"]
        row = dict(head=head, contexts_attempted=len(group), contexts_passed=len(good),
                   contexts_failed=len(group)-len(good))
        if good:
            for prefix, name in [("projected", "literal_update_query_variation"),
                                 ("linear", "linear_update_query_variation")]:
                numerator = sum(r[prefix+"_matrix_variation_energy"] for r in good)
                denominator = sum(r[prefix+"_matrix_energy"] for r in good)
                row[name] = section7a_ratio(numerator, denominator, root=True)
            for key in ["frozen_update_transfer_mse", "constant_output_baseline_mse", "mean_value_baseline_mse"]:
                row[key] = float(np.mean([r[key] for r in good]))
            ratio = section7a_ratio(row["frozen_update_transfer_mse"], row["constant_output_baseline_mse"])
            row["frozen_transfer_relative_rmse"] = None if ratio is None else math.sqrt(ratio)
            row["frozen_transfer_r2_vs_constant"] = None if ratio is None else 1-ratio
            values = [r["literal_update_query_variation"] for r in good if r["literal_update_query_variation"] is not None]
            row["context_median_query_variation"] = float(np.median(values)) if values else None
            # Descriptive context dispersion, not a query-level significance test.
            mse = [r["frozen_update_transfer_mse"] for r in good]
            row["context_sd_transfer_mse"] = float(np.std(mse, ddof=1)) if len(mse) > 1 else None
            row["output_max_abs_error"] = max(r["output_max_abs_error"] for r in good)
            if all("adaptive_projected_mse" in r for r in good):
                row["adaptive_projected_mse"] = float(np.mean([r["adaptive_projected_mse"] for r in good]))
        summary.append(row)
    return summary


def section7a_write_csv(path, rows):
    fields = list(dict.fromkeys(key for row in rows for key in row))
    write_csv(path, rows, fields)


def section7a_refresh(query, left, keys, values, saved_fast=None, anchor=0):
    """Restore the exact term omitted by freezing a query-dependent matrix.

    L = Wq_aug R_query^T / sqrt(d), c_ij = rho(s_ij)/Z_i.
    T_i = sum_j c_ij k_j (v_j-mu); DeltaM_i = L T_i.
    T_i-T_0 is recomputed from c_i-c_0, not fitted to target outputs.
    y_i = mu + x_i L T_0 + x_i L (T_i-T_0).
    This is an exact readout, not a context-only constant matrix or speedup.
    """
    query, left, keys, values = [np.asarray(x, dtype=np.float64) for x in (query, left, keys, values)]
    if (query.ndim != 2 or left.ndim != 2 or keys.ndim != 2 or values.ndim != 2
            or query.shape[1] != left.shape[0] or left.shape[1] != keys.shape[1]
            or len(keys) != len(values) or not len(keys) or not 0 <= anchor < len(query)):
        raise ValueError("Invalid saved-factor dimensions or anchor index.")
    if not all(np.isfinite(x).all() for x in (query, left, keys, values)):
        raise FloatingPointError("Nonfinite inputs to exact query refresh.")
    qscaled = query@left
    scores = qscaled@keys.T
    coef = section7a_coefficients(scores)
    shifted = scores-scores.max(1, keepdims=True)
    probability = np.exp(shifted); probability /= probability.sum(1, keepdims=True)
    mu = values.mean(0)
    centered = values-mu
    fast = keys.T[None]@(coef[..., None]*centered[None])
    if saved_fast is not None:
        np.testing.assert_allclose(fast, saved_fast, atol=2e-9, rtol=2e-8,
            err_msg="Saved T is not the original-score Section 7A matrix factor.")
    # Independently reconstruct the difference directly from coefficient changes.
    delta_fast = keys.T[None]@((coef-coef[anchor])[..., None]*centered[None])
    np.testing.assert_allclose(fast[anchor]+delta_fast, fast, atol=2e-9, rtol=2e-8)
    frozen = mu+qscaled@fast[anchor]
    correction = np.einsum("qd,qdr->qr", qscaled, delta_fast)
    exact = frozen+correction
    literal = mu+np.einsum("qd,qdr->qr", qscaled, fast)
    direct = probability@values
    np.testing.assert_allclose(literal, direct, atol=2e-9, rtol=2e-8,
                               err_msg="Per-query literal identity failed.")
    np.testing.assert_allclose(exact, direct, atol=2e-9, rtol=2e-8,
                               err_msg="Anchor plus exact query-dependent correction failed.")
    return dict(left=left, fast=fast, delta_fast=delta_fast, mu=mu, scores=scores,
        literal=literal, exact=exact, frozen=frozen, correction=correction, direct=direct)


def section7a_full_readout(query, left, keys, values, wk, wv, cosine, sine):
    """Literal Section 7A for the native final row, including its own self K/V.

    The value mean is now mu_i=(sum(prefix_values)+self_value_i)/(N+1).
    The self key/value may depend on the query: the identity is pointwise and
    remains exact. This function makes no frozen-bank Jacobian assertion.
    """
    query, left, keys, values, wk, wv, cosine, sine = [np.asarray(x, dtype=np.float64)
        for x in (query, left, keys, values, wk, wv, cosine, sine)]
    if query.ndim != 2 or len(query) == 0 or not len(keys) or len(values) != len(keys):
        raise ValueError("Full readout requires nonempty queries and prefix K/V.")
    if not all(np.isfinite(x).all() for x in (query, left, keys, values, wk, wv, cosine, sine)):
        raise FloatingPointError("Nonfinite full-attention inputs.")
    qscaled = query@left
    self_key = section7a_rotate(query@wk, cosine, sine)
    self_value = query@wv
    prefix_scores = qscaled@keys.T
    self_score = np.einsum("qd,qd->q", qscaled, self_key)
    scores = np.column_stack((prefix_scores, self_score))
    coef = section7a_coefficients(scores)
    probability = np.exp(scores-scores.max(1, keepdims=True))
    probability /= probability.sum(1, keepdims=True)
    mu = (values.sum(0)[None]+self_value)/(len(values)+1)
    weighted_prefix = coef[:, :-1, None]*(values[None]-mu[:, None])
    fast = keys.T[None]@weighted_prefix
    fast += coef[:, -1, None, None]*self_key[:, :, None]*(self_value-mu)[:, None, :]
    literal = mu+np.einsum("qd,qdr->qr", qscaled, fast)
    direct = probability[:, :-1]@values+probability[:, -1:]*self_value
    np.testing.assert_allclose(literal, direct, atol=2e-9, rtol=2e-8,
                               err_msg="Full native-bank literal identity failed.")
    return dict(fast=fast, mu=mu, literal=literal, direct=direct,
                self_value=self_value, self_mass=probability[:, -1:],
                prefix_mass=probability[:, :-1].sum(1, keepdims=True))


def section7a_equivalence_metrics(prefix, full, wo):
    """Independent exact checks and named approximate controls in output space."""
    outputs = {name: prefix[name]@wo for name in ["literal", "exact", "frozen", "correction", "direct"]}
    outputs["full_literal"] = full["literal"]@wo
    outputs["full_direct"] = full["direct"]@wo
    outputs["full_exact"] = (full["prefix_mass"]*prefix["exact"]+full["self_mass"]*full["self_value"])@wo
    # These are explicitly APPROXIMATE controls with native self routing kept.
    outputs["full_frozen_prefix"] = (full["prefix_mass"]*prefix["frozen"]+full["self_mass"]*full["self_value"])@wo
    outputs["full_constant_prefix"] = (full["prefix_mass"]*prefix["direct"][:1]+full["self_mass"]*full["self_value"])@wo
    np.testing.assert_allclose(outputs["full_exact"], outputs["full_direct"], atol=3e-9, rtol=3e-8)
    return section7a_output_metrics(outputs), outputs


def section7a_output_metrics(outputs):
    if not all(np.isfinite(value).all() for value in outputs.values()):
        raise FloatingPointError("Nonfinite projected readout; no passing identity metric can be reported.")
    metrics = {}
    comparisons = [("prefix_literal", "literal", "direct"),
                   ("prefix_anchor_plus_refresh", "exact", "direct"),
                   ("full_literal", "full_literal", "full_direct"),
                   ("full_anchor_plus_refresh", "full_exact", "full_direct")]
    for label, actual, expected in comparisons:
        error = outputs[actual]-outputs[expected]
        metrics[label+"_mse"] = float(np.mean(error**2))
        metrics[label+"_max_abs_error"] = float(np.max(np.abs(error)))
    missed = outputs["direct"]-outputs["frozen"]-outputs["correction"]
    metrics["frozen_residual_equals_omitted_term_max_abs_error"] = float(np.max(np.abs(missed)))
    metrics["prefix_frozen_mse"] = float(np.mean((outputs["frozen"][1:]-outputs["direct"][1:])**2))
    metrics["prefix_constant_baseline_mse"] = float(np.mean((outputs["direct"][:1]-outputs["direct"][1:])**2))
    metrics["prefix_omitted_correction_mse"] = float(np.mean(outputs["correction"][1:]**2))
    np.testing.assert_allclose(metrics["prefix_omitted_correction_mse"], metrics["prefix_frozen_mse"],
                               atol=3e-9, rtol=3e-8,
                               err_msg="Frozen error is not explained by the omitted query-dependent term.")
    metrics["full_frozen_prefix_mse"] = float(np.mean((outputs["full_frozen_prefix"][1:]-outputs["full_direct"][1:])**2))
    metrics["full_constant_prefix_baseline_mse"] = float(np.mean((outputs["full_constant_prefix"][1:]-outputs["full_direct"][1:])**2))
    for label in ["prefix", "full"]:
        error = metrics["prefix_frozen_mse" if label == "prefix" else "full_frozen_prefix_mse"]
        baseline = metrics["prefix_constant_baseline_mse" if label == "prefix" else "full_constant_prefix_baseline_mse"]
        metrics[label+"_frozen_relative_rmse"] = section7a_ratio(error, baseline, root=True)
    return metrics


def section7a_replay(source, output):
    """NumPy-only replay of recorded pretrained activations; no new inference."""
    source, output = Path(source).resolve(), Path(output).resolve()
    files = sorted(source.glob("context_*.npz"))
    required = [source/"weights.npz", source/"metadata.json"]
    if not files or any(not p.is_file() for p in required):
        raise FileNotFoundError("Replay needs weights.npz, metadata.json, and context_*.npz from the previous run; diagnostics.csv alone cannot reconstruct missing query terms.")
    if output.exists() and any(output.iterdir()):
        raise FileExistsError("Replay requires a new or empty --out directory; previous results are preserved.")
    output.mkdir(parents=True, exist_ok=True)
    original_meta = json.loads((source/"metadata.json").read_text())
    with np.load(source/"weights.npz", allow_pickle=False) as saved:
        weights = {name: saved[name].copy() for name in saved.files}
    wq, wk, wv, wo = [weights[name] for name in ["Wq_aug", "Wk_aug", "Wv_aug", "Wo"]]
    H, m, d = wq.shape
    mapping = weights["head_to_kv"].astype(int)
    if mapping.shape != (H,) or not np.array_equal(mapping, np.arange(H)//(H//len(wk))):
        raise ValueError("Invalid recorded Qwen GQA head mapping.")
    old_rows = {}
    if (source/"diagnostics.csv").is_file():
        with (source/"diagnostics.csv").open() as f:
            old_rows = {(r["context"], r["head"]): r for r in csv.DictReader(f)}
    metadata = dict(protocol_version=5, experiment="section7a_exact_query_refresh_replay",
        status="running", source_directory=str(source), source_protocol_version=original_meta.get("protocol_version"),
        source_model=original_meta.get("model"), source_layer=original_meta.get("layer"),
        source_model_commit=original_meta.get("model_commit"),
        checkpoint_kind=original_meta.get("checkpoint_kind", "unspecified"),
        matrix_convention="section7a_original_scores_homogeneous", dtype="float64",
        exact_claim="Each query uses its own original-score coefficients; full readout includes its own self K/V and the correct value mean.",
        frozen_control="An approximation. Its error is the query-contracted change of the effective matrix, verified explicitly.",
        requires_new_model_inference=False, fitted_parameters=0, removed_heads=[],
        source_metadata_sha256=hashlib.sha256((source/"metadata.json").read_bytes()).hexdigest(),
        source_weights_sha256=hashlib.sha256((source/"weights.npz").read_bytes()).hexdigest(),
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), contexts=[])
    rows, per_query = [], []

    def save():
        section7a_write_csv(output/"equivalence.csv", rows)
        section7a_write_csv(output/"per_query.csv", per_query)
        (output/"metadata.json").write_text(json.dumps(metadata, indent=2, allow_nan=False))

    save()
    for index, filename in enumerate(files):
        context_id = filename.stem.removeprefix("context_").lstrip("0") or "0"
        context_rows, heads_saved, failed = [], {}, False
        try:
            with np.load(filename, allow_pickle=False) as saved:
                data = {name: saved[name].copy() for name in saved.files}
            query = data["query_aug"]
            if query.ndim != 2 or query.shape[1] != m or len(query) < 2 or not np.all(query[:, -1] == 1):
                raise ValueError("Expected at least two normalized query activations with homogeneous coordinate 1.")
            prefix_hidden = data["prefix_hidden_aug"]
            if prefix_hidden.ndim != 2 or prefix_hidden.shape[1] != m or not np.all(prefix_hidden[:, -1] == 1):
                raise ValueError("Recorded prefix inputs do not use the expected homogeneous coordinates.")
            checked_keys = section7a_rotate(prefix_hidden[None]@wk, data["prefix_cos"][None], data["prefix_sin"][None])
            checked_values = prefix_hidden[None]@wv
            np.testing.assert_allclose(checked_keys, data["keys"], atol=2e-12, rtol=2e-11,
                err_msg="Recorded keys do not match the recorded affine projections and RoPE.")
            np.testing.assert_allclose(checked_values, data["values"], atol=2e-12, rtol=2e-11,
                err_msg="Recorded values do not match the recorded affine projection.")
            total = None
            for head in range(H):
                kv = int(mapping[head])
                row = dict(context=int(context_id), head=head, queries=len(query),
                    status="passed", frozen_control="approximation", bank="prefix_and_native_full")
                try:
                    left = section7a_rotate(wq[head], data["query_cos"], data["query_sin"])/math.sqrt(d)
                    np.testing.assert_allclose(left, data["L"][head], atol=2e-12, rtol=2e-11,
                        err_msg="Recorded L does not match the literal biased query projection.")
                    prefix = section7a_refresh(query, left, data["keys"][kv], data["values"][kv], data["T"][head])
                    full = section7a_full_readout(query, left, data["keys"][kv], data["values"][kv],
                                                wk[kv], wv[kv], data["query_cos"], data["query_sin"])
                    metrics, outputs = section7a_equivalence_metrics(prefix, full, wo[head])
                    row.update(metrics)
                    row["native_self_attention_mass_mean"] = float(full["self_mass"].mean())
                    np.testing.assert_allclose(full["self_mass"][:, 0], data["native_self_attention_mass"][:, head], atol=3e-5, rtol=3e-4)
                    old = old_rows.get((context_id, str(head)))
                    if old and old.get("frozen_update_transfer_mse"):
                        old_error = float(old["frozen_update_transfer_mse"])
                        np.testing.assert_allclose(metrics["prefix_frozen_mse"], old_error, atol=3e-9, rtol=3e-8,
                            err_msg="Frozen control differs from the original run; the failed control must be preserved.")
                        row["original_frozen_control_mse"] = old_error
                        row["original_literal_matrix_variation"] = float(old["literal_update_query_variation"])
                    if total is None:
                        total = {name: value.copy() for name, value in outputs.items()}
                    else:
                        for name in total:
                            total[name] += outputs[name]
                    heads_saved[f"h{head}_prefix_T"] = prefix["fast"]
                    heads_saved[f"h{head}_delta_T"] = prefix["delta_fast"]
                    heads_saved[f"h{head}_full_T"] = full["fast"]
                    heads_saved[f"h{head}_full_mu"] = full["mu"]
                    for qi in range(len(query)):
                        per_query.append(dict(context=int(context_id), head=head, query=qi, is_anchor=qi == 0,
                            prefix_exact_mse=float(np.mean((outputs["exact"][qi]-outputs["direct"][qi])**2)),
                            prefix_frozen_mse=float(np.mean((outputs["frozen"][qi]-outputs["direct"][qi])**2)),
                            full_exact_mse=float(np.mean((outputs["full_exact"][qi]-outputs["full_direct"][qi])**2))))
                except (AssertionError, FloatingPointError, ValueError) as exc:
                    row.update(status="validation_failed", error=str(exc))
                    failed = True
                context_rows.append(row)
            block = dict(context=int(context_id), head="all", queries=len(query), status="passed", frozen_control="approximation", bank="prefix_and_native_full")
            if failed:
                block.update(status="validation_failed", error="One or more heads failed; no partial-head total was substituted.")
            else:
                block.update(section7a_output_metrics(total))
                for name, saved_name in [("direct", "conditional_output"), ("frozen", "frozen_output")]:
                    np.testing.assert_allclose(total[name], data[saved_name], atol=3e-9, rtol=3e-8,
                        err_msg="Replayed outputs differ from the original recording.")
                for name in ["full_literal", "full_exact"]:
                    np.testing.assert_allclose(total[name], data["native_full_attention_output"], atol=3e-5, rtol=3e-4,
                        err_msg="Full literal replay does not match recorded native Qwen output.")
                block["native_fp32_max_abs_error"] = float(np.max(np.abs(total["full_literal"]-data["native_full_attention_output"])))
                block["native_fp32_mse"] = float(np.mean((total["full_literal"]-data["native_full_attention_output"])**2))
                old = old_rows.get((context_id, "all"))
                if old and old.get("frozen_update_transfer_mse"):
                    old_error = float(old["frozen_update_transfer_mse"])
                    np.testing.assert_allclose(block["prefix_frozen_mse"], old_error, atol=3e-9, rtol=3e-8)
                    block["original_frozen_control_mse"] = old_error
                    block["original_literal_matrix_variation"] = float(old["literal_update_query_variation"])
                for qi in range(len(query)):
                    per_query.append(dict(context=int(context_id), head="all", query=qi, is_anchor=qi == 0,
                        prefix_exact_mse=float(np.mean((total["exact"][qi]-total["direct"][qi])**2)),
                        prefix_frozen_mse=float(np.mean((total["frozen"][qi]-total["direct"][qi])**2)),
                        full_exact_mse=float(np.mean((total["full_exact"][qi]-total["full_direct"][qi])**2))))
                np.savez_compressed(output/f"replay_{int(context_id):04d}.npz", **total, **heads_saved,
                                    query_aug=query, L=data["L"], token_ids=data["query_token_ids"])
            context_rows.append(block)
        except Exception as exc:
            failed = True
            context_rows.append(dict(context=int(context_id), head="all", status="validation_failed", error=str(exc)))
        rows.extend(context_rows)
        metadata["contexts"].append(dict(context=int(context_id), source_file=filename.name,
            source_sha256=hashlib.sha256(filename.read_bytes()).hexdigest(), status="failed" if failed else "passed"))
        save()
        if failed:
            print(f"Replay {index+1}/{len(files)}: VALIDATION FAILED; see equivalence.csv.", flush=True)
        else:
            print(f"Replay {index+1}/{len(files)}: exact prefix MSE={block['prefix_anchor_plus_refresh_mse']:.2e}; "
                f"exact full MSE={block['full_literal_mse']:.2e}; native fp32 error={block['native_fp32_max_abs_error']:.2e}; "
                f"FROZEN control MSE={block['prefix_frozen_mse']:.6g}", flush=True)
    metadata["status"] = "completed" if all(row["status"] == "passed" for row in rows) else "completed_with_validation_failures"
    summary = []
    for head in list(range(H))+["all"]:
        group = [r for r in rows if r["head"] == head]
        good = [r for r in group if r["status"] == "passed"]
        entry = dict(head=head, contexts_passed=len(good), contexts_attempted=len(group))
        if good:
            for key in good[0]:
                if key.endswith("_mse") and all(key in r for r in good):
                    entry[key] = float(np.mean([r[key] for r in good]))
                elif key.endswith("_max_abs_error") and all(key in r for r in good):
                    entry[key] = max(r[key] for r in good)
            entry["prefix_frozen_relative_rmse"] = section7a_ratio(entry["prefix_frozen_mse"], entry["prefix_constant_baseline_mse"], root=True)
            entry["full_frozen_relative_rmse"] = section7a_ratio(entry["full_frozen_prefix_mse"], entry["full_constant_prefix_baseline_mse"], root=True)
        summary.append(entry)
    metadata["summary"] = summary
    section7a_write_csv(output/"heads.csv", summary)
    save()
    lines = ["# Exact equivalence and frozen-matrix approximation", "",
        "This replay uses the recorded activations and projections. It performs no training, fitting, head selection, or new model forward pass. Random fixture/smoke results are not pretrained evidence; checkpoint kind: "+str(metadata["checkpoint_kind"])+".", "",
        "The exact identity uses DeltaM_i for each query i. The frozen control instead uses DeltaM_0. With one fixed prefix, its omitted term is x_i^T(DeltaM_i-DeltaM_0). A large error in that control is compatible with an exact original identity.", "",
        "Write L=Wq_aug R_query^T/sqrt(d), c_ij=rho(s_ij)/Z_i, T_i=sum_j c_ij k_j(v_j-mu). Then DeltaM_i=L T_i. The code reconstructs T_i-T_0 directly from c_i-c_0 and checks y_i=mu+x_i^T L T_0+x_i^T L(T_i-T_0) against an independent softmax readout.", "",
        "Native full attention also includes the current query's self key/value. The replay recomputes its literal full-bank T_i and mean mu_i. Independently, it checks y_full=(1-self_mass)*y_prefix+self_mass*v_self. The recorded native fp32 output is compared separately from float64 algebraic error.", "",
        "The full frozen-prefix control keeps native self routing and approximates only the prefix readout. It is explicitly an approximation with no exactness guarantee. Neither this control nor coefficient refreshing establishes a constant shared matrix or an inference speedup.", "",
        "| Head | Passed/attempted | Exact prefix MSE | Exact full MSE | Frozen prefix MSE | Frozen/constant RMSE |",
        "|---|---:|---:|---:|---:|---:|"]
    def fmt(x):
        return "undefined" if x is None else f"{x:.6g}"
    for row in summary:
        lines.append(f"| {row['head']} | {row['contexts_passed']}/{row['contexts_attempted']} | "+
            " | ".join(fmt(row.get(k)) for k in ["prefix_anchor_plus_refresh_mse", "full_literal_mse", "prefix_frozen_mse", "prefix_frozen_relative_rmse"])+" |")
    lines += ["", "All MSEs are in the projected attention-output coordinates. Frozen controls exclude anchor query 0. Full-head results sum outputs before computing errors and therefore retain cross-head terms. Context summaries average context MSEs; no query-level significance test is performed.", "",
        "The uploaded diagnostics alone already contain exact reconstruction checks. Replaying requires the original weights.npz, metadata.json, and context_*.npz; a CSV cannot determine missing query-specific corrections.", "",
        "Files: equivalence.csv records each context/head; heads.csv pools contexts; per_query.csv gives query errors; replay_*.npz stores the recomputed factors and outputs; metadata.json records source hashes and any validation failures. Original frozen-control errors are checked and retained."]
    (output/"REPORT.md").write_text("\n".join(lines)+"\n")
    print(f"{metadata['status']}: exact reconstruction and frozen approximation are reported separately in {output}", flush=True)
    return metadata


def shared_update_probe(args, model, tokenizer, module, rotary, layer, load_blocks, data_info, output):
    """Read-only native hook plus an explicit fixed-prefix conditional readout."""
    import torch

    cfg = model.config
    if cfg.model_type != "qwen2" or hasattr(module, "q_norm") or hasattr(module, "k_norm"):
        raise ValueError("--shared-update supports affine Qwen2 attention only; post-projection Q/K norms require a different derivation.")
    if min(args.contexts, args.queries-1, args.batch) < 1:
        raise ValueError("Need contexts >= 1, queries >= 2, and batch >= 1.")
    if args.queries >= cfg.vocab_size:
        raise ValueError("--queries must be smaller than the vocabulary.")
    if any(output.iterdir()):
        raise FileExistsError("--shared-update requires a new or empty --out directory to avoid mixing experiments.")
    device = next(model.parameters()).device
    H, HK = cfg.num_attention_heads, cfg.num_key_value_heads
    D = getattr(cfg, "head_dim", None) or cfg.hidden_size//H
    groups = H//HK
    scale = float(getattr(module, "scaling", D**-.5))
    if not math.isclose(scale, D**-.5, rel_tol=1e-12):
        raise ValueError("Unexpected attention scaling; this probe specifies 1/sqrt(head_dim).")
    if getattr(module.o_proj, "bias", None) is not None:
        raise ValueError("This Qwen probe expects a bias-free output projection.")

    def array(tensor):
        return tensor.detach().cpu().double().numpy()

    def affine(projection, heads):
        weight = array(projection.weight).T
        bias = np.zeros(weight.shape[1]) if projection.bias is None else array(projection.bias)
        return np.concatenate((weight, bias[None]), axis=0).reshape(weight.shape[0]+1, heads, D).transpose(1, 0, 2)

    wq, wk, wv = affine(module.q_proj, H), affine(module.k_proj, HK), affine(module.v_proj, HK)
    wo = array(module.o_proj.weight).T.reshape(H, D, cfg.hidden_size)
    np.savez_compressed(output/"weights.npz", Wq_aug=wq, Wk_aug=wk, Wv_aug=wv, Wo=wo,
                        inv_freq=array(rotary.inv_freq), head_to_kv=np.arange(H)//groups)
    metadata = dict(protocol_version=5, experiment="pretrained_section7a_fixed_prefix",
        matrix_convention="section7a_original_scores_homogeneous", arguments=vars(args),
        model=args.model, model_commit=getattr(cfg, "_commit_hash", None), model_config=cfg.to_dict(),
        layer=layer, heads=H, kv_heads=HK, head_dim=D, device=str(device),
        native_dtype="float32", diagnostic_dtype="float64", status="running",
        versions={name: importlib.metadata.version(name) for name in ["torch", "transformers", "numpy"]},
        checkpoint_kind="random_smoke_model" if args.smoke else "pretrained",
        gauge="Original uncentered logits; homogeneous coordinate fixed at 1; no key shift, fit, or post-hoc rescaling.",
        coordinate="Input to q_proj/k_proj/v_proj after input RMSNorm, augmented by 1; not the raw residual stream.",
        query_sampling="Query 0 is the observed next token; other queries are highest-ranked distinct non-special prefix continuations.",
        attended_bank="Prefix only, identical for every query; current query self K/V excluded from the conditional measurement.",
        native_validation="Read-only hook verifies the complete final native attention row including its own self K/V.",
        transfer="Freeze literal DeltaM at observed query 0; evaluate all other queries without fitting.",
        interpretation="Conditional frozen-prefix attention measurement; not an end-to-end intervention, GD claim, or demonstration of novelty.",
        source="https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen2/modeling_qwen2.py",
        script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), contexts=[])
    all_rows, query_rows = [], []

    def save_metadata():
        (output/"metadata.json").write_text(json.dumps(metadata, indent=2, allow_nan=False))

    save_metadata()
    capture = {}

    def hook(mod, positional, kw, native):
        hs = kw.get("hidden_states")
        if hs is None:
            hs = positional[0]
        b, length, _ = hs.shape
        pe = kw.get("position_embeddings")
        if pe is None:
            pe = rotary(hs, torch.arange(length, device=hs.device)[None])
        cosine, sine = pe
        if cosine.ndim != 3 or cosine.shape[1:] != (length, D) or sine.shape != cosine.shape:
            raise ValueError("Expected unpadded [batch or 1, sequence, head_dim] position embeddings.")
        cosine, sine = cosine.expand(b, -1, -1), sine.expand(b, -1, -1)
        q = mod.q_proj(hs).reshape(b, length, H, D).transpose(1, 2)
        k = mod.k_proj(hs).reshape(b, length, HK, D).transpose(1, 2)
        v = mod.v_proj(hs).reshape(b, length, HK, D).transpose(1, 2)

        def rotate(t):
            half = torch.cat((-t[..., D//2:], t[..., :D//2]), dim=-1)
            return t*cosine[:, None]+half*sine[:, None]

        q, k = rotate(q), rotate(k)
        expanded_k, expanded_v = k.repeat_interleave(groups, 1), v.repeat_interleave(groups, 1)
        scores = (q[:, :, -1:]@expanded_k.transpose(-1, -2))*scale
        mask = kw.get("attention_mask")
        if mask is not None:
            if mask.dtype == torch.bool or mask.ndim != 4:
                raise ValueError("Expected an additive 4-D causal mask, or None.")
            final_mask = mask[..., -1:, :length]
            if torch.any(final_mask != 0):
                raise ValueError("The final query must attend every sequence entry; padding/additional score biases are unsupported.")
        probability = scores.softmax(-1)
        head_output = (probability@expanded_v)[:, :, 0]
        rebuilt = mod.o_proj(head_output.reshape(b, H*D))
        ref = native[0] if isinstance(native, tuple) else native
        torch.testing.assert_close(rebuilt, ref[:, -1], atol=2e-5, rtol=2e-4,
                                   msg="Native Qwen final-row reconstruction failed.")
        capture.clear()
        capture.update(hidden=array(hs), cosine=array(cosine), sine=array(sine),
            q=array(q[:, :, -1]), k=array(k), v=array(v),
            native=array(ref[:, -1]), native_head=array(head_output),
            self_mass=array(probability[:, :, 0, -1]),
            native_error=float((rebuilt-ref[:, -1]).abs().max().item()))
        return native

    handle = module.register_forward_hook(hook, with_kwargs=True)
    try:
        blocks = load_blocks("test", args.contexts)
        metadata["data"] = data_info
        save_metadata()
        with torch.inference_mode():
            for context_index, block in enumerate(blocks):
                prefix, observed = block[:-1].to(device), int(block[-1])
                # Use core + the last LM-head row, avoiding a length-by-vocabulary tensor.
                hidden = model.model(input_ids=prefix[None], use_cache=False, return_dict=True).last_hidden_state[:, -1]
                logits = model.lm_head(hidden)[0].float()
                banned = set([] if tokenizer is None else tokenizer.all_special_ids)
                eligible_logits = logits.clone()
                banned_ids = [t for t in banned if 0 <= t < len(logits)]
                if banned_ids:
                    eligible_logits[banned_ids] = -float("inf")
                eligible_logits[observed] = -float("inf")
                if int(torch.isfinite(eligible_logits).sum()) < args.queries-1:
                    raise ValueError("Not enough eligible distinct continuations for --queries.")
                candidates = [observed]+torch.topk(eligible_logits, args.queries-1).indices.cpu().tolist()
                candidate_prob = logits.softmax(0)[candidates].cpu().double().tolist()
                captured_hidden, captured_q, captured_native, captured_native_head, self_mass = [], [], [], [], []
                context = None
                native_error, prefix_error = 0., 0.
                for start in range(0, args.queries, args.batch):
                    selected = torch.tensor(candidates[start:start+args.batch], device=device, dtype=torch.long)
                    ids = torch.cat((prefix[None].expand(len(selected), -1), selected[:, None]), dim=1)
                    model.model(input_ids=ids, use_cache=False, return_dict=True)
                    item = capture
                    if context is None:
                        context = {name: item[name][0, :-1].copy() for name in ["hidden", "cosine", "sine"]}
                        context["qcos"], context["qsin"] = item["cosine"][0, -1].copy(), item["sine"][0, -1].copy()
                        context["native_k"], context["native_v"] = item["k"][0, :, :-1].copy(), item["v"][0, :, :-1].copy()
                    for j in range(len(selected)):
                        for name in ["hidden", "cosine", "sine"]:
                            comparison = item[name][j, :-1]
                            np.testing.assert_allclose(comparison, context[name], atol=2e-5, rtol=2e-4,
                                                       err_msg="Prefix changed across query branches.")
                            prefix_error = max(prefix_error, float(np.max(np.abs(comparison-context[name]))))
                        np.testing.assert_array_equal(item["cosine"][j, -1], context["qcos"])
                        np.testing.assert_array_equal(item["sine"][j, -1], context["qsin"])
                    captured_hidden.append(item["hidden"][:, -1].copy())
                    captured_q.append(item["q"].copy())
                    captured_native.append(item["native"].copy())
                    captured_native_head.append(item["native_head"].copy())
                    self_mass.append(item["self_mass"].copy())
                    native_error = max(native_error, item["native_error"])
                query_hidden = np.concatenate(captured_hidden)
                query = np.concatenate((query_hidden, np.ones((len(query_hidden), 1))), axis=1)
                prefix_hidden = np.concatenate((context["hidden"], np.ones((len(prefix), 1))), axis=1)
                native_q, native_output = np.concatenate(captured_q), np.concatenate(captured_native)
                native_head, self_mass = np.concatenate(captured_native_head), np.concatenate(self_mass)
                keys = section7a_rotate(prefix_hidden[None]@wk, context["cosine"][None], context["sine"][None])
                values = prefix_hidden[None]@wv
                np.testing.assert_allclose(keys, context["native_k"], atol=2e-5, rtol=2e-4)
                np.testing.assert_allclose(values, context["native_v"], atol=2e-5, rtol=2e-4)
                projected_native = np.zeros((args.queries, cfg.hidden_size))
                projected_adaptive = np.zeros_like(projected_native)
                projected_frozen = np.zeros_like(projected_native)
                projected_mu = np.zeros(cfg.hidden_size)
                lefts, fasts, means, dense = [], [], [], []
                context_rows = []
                for head in range(H):
                    kv = head//groups
                    row = dict(context=context_index, layer=layer, head=head, kv_head=kv,
                        queries=args.queries, prefix_length=len(prefix), query_position=len(prefix),
                        status="passed", matrix_convention="section7a_original_scores_homogeneous",
                        native_hook_max_abs_error=native_error, prefix_max_abs_difference=prefix_error,
                        native_self_attention_mass_mean=float(self_mass[:, head].mean()))
                    try:
                        result = section7a_effective(query, wq[head], keys[kv], values[kv], context["qcos"], context["qsin"], scale)
                        np.testing.assert_allclose(query@result["left"]/scale, native_q[:, head], atol=2e-5, rtol=2e-4)
                        # Independently include each branch's own self entry to match native Qwen.
                        self_key = section7a_rotate(query@wk[kv], context["qcos"], context["qsin"])
                        self_value = query@wv[kv]
                        self_score = np.einsum("qd,qd->q", query@result["left"], self_key)
                        full_scores = np.column_stack((result["scores"], self_score))
                        full_p = np.exp(full_scores-full_scores.max(1, keepdims=True))
                        full_p /= full_p.sum(1, keepdims=True)
                        full_native = full_p[:, :-1]@values[kv]+full_p[:, -1:]*self_value
                        np.testing.assert_allclose(full_native, native_head[:, head], atol=3e-5, rtol=3e-4,
                            err_msg="Float64 affine/RoPE formula failed native head reconstruction.")
                        row["native_float64_head_max_abs_error"] = float(np.max(np.abs(full_native-native_head[:, head])))
                        stats, y, frozen, mu = section7a_head_statistics(result, query, wo[head])
                        # Each query uses its own DeltaM_i, including its bias row.
                        adaptive_head = result["mu"] + np.einsum("qm,qmd->qd", query, result["matrix"])
                        adaptive_output = adaptive_head @ wo[head]
                        np.testing.assert_allclose(adaptive_output, y, atol=3e-9, rtol=3e-8,
                            err_msg="Per-query effective-matrix readout differs from direct attention.")
                        row.update(stats)
                        row["adaptive_projected_mse"] = float(np.mean((adaptive_output-y)**2))
                        projected_native += y
                        projected_adaptive += adaptive_output
                        projected_frozen += frozen
                        projected_mu += mu
                        lefts.append(result["left"])
                        fasts.append(result["fast"])
                        means.append(result["mu"])
                        if args.save_matrices:
                            dense.append(result["matrix"])
                        for qi in range(args.queries):
                            query_rows.append(dict(context=context_index, layer=layer, head=head, query=qi,
                                token_id=candidates[qi], is_anchor=qi == 0, prefix_probability=candidate_prob[qi],
                                adaptive_projected_mse=float(np.mean((adaptive_output[qi]-y[qi])**2)),
                                frozen_projected_mse=float(np.mean((frozen[qi]-y[qi])**2)),
                                constant_output_mse=float(np.mean((y[0]-y[qi])**2)),
                                native_self_attention_mass=float(self_mass[qi, head])))
                    except (AssertionError, FloatingPointError) as exc:
                        row.update(status="numerical_validation_failed", error=str(exc))
                    context_rows.append(row)
                block_row = dict(context=context_index, layer=layer, head="all", kv_head="all", queries=args.queries,
                    prefix_length=len(prefix), query_position=len(prefix), status="passed",
                    matrix_convention="section7a_original_scores_homogeneous",
                    native_hook_max_abs_error=native_error, prefix_max_abs_difference=prefix_error)
                if len(lefts) == H:
                    lefts, fasts, means = np.stack(lefts), np.stack(fasts), np.stack(means)
                    # Streaming Welford variance of SUM_h DeltaM_h Wo_h; cross-head
                    # terms are retained. Only one full block matrix lives at a time.
                    tl = torch.as_tensor(lefts, device=device, dtype=torch.float64)
                    tf = torch.as_tensor(fasts, device=device, dtype=torch.float64)
                    to = torch.as_tensor(wo, device=device, dtype=torch.float64)
                    tq = torch.as_tensor(query, device=device, dtype=torch.float64)
                    tmu = torch.as_tensor(projected_mu, device=device, dtype=torch.float64)
                    ty = torch.as_tensor(projected_native, device=device, dtype=torch.float64)
                    running_mean = torch.zeros((query.shape[1], cfg.hidden_size), device=device, dtype=torch.float64)
                    total_energy = variance_energy = linear_energy = linear_variance = 0.
                    block_error = 0.
                    for qi in range(args.queries):
                        update = torch.bmm(torch.bmm(tl, tf[:, qi]), to).sum(0)
                        rebuilt = tq[qi]@update+tmu
                        torch.testing.assert_close(rebuilt, ty[qi], atol=3e-9, rtol=3e-8,
                                                   msg="Sum of literal head matrices failed block reconstruction.")
                        block_error = max(block_error, float((rebuilt-ty[qi]).abs().max().item()))
                        old_delta = update-running_mean
                        running_mean += old_delta/(qi+1)
                        increment = old_delta*(update-running_mean)
                        variance_energy += float(increment.sum().item())
                        linear_variance += float(increment[:-1].sum().item())
                        total_energy += float(update.square().sum().item())
                        linear_energy += float(update[:-1].square().sum().item())
                    block_row.update(projected_matrix_energy=total_energy,
                        projected_matrix_variation_energy=max(variance_energy, 0.),
                        linear_matrix_energy=linear_energy, linear_matrix_variation_energy=max(linear_variance, 0.),
                        literal_update_query_variation=section7a_ratio(variance_energy, total_energy, root=True),
                        linear_update_query_variation=section7a_ratio(linear_variance, linear_energy, root=True),
                        output_max_abs_error=block_error,
                        adaptive_projected_mse=float(np.mean((projected_adaptive-projected_native)**2)),
                        **section7a_transfer(projected_native, projected_frozen, projected_mu))
                    arrays = dict(L=lefts, T=fasts, mu=means, query_aug=query, prefix_hidden_aug=prefix_hidden,
                        prefix_ids=prefix.cpu().numpy(), query_token_ids=np.array(candidates),
                        prefix_continuation_probability=np.array(candidate_prob), key_positions=np.arange(len(prefix)),
                        query_position=np.array(len(prefix)), query_cos=context["qcos"], query_sin=context["qsin"],
                        prefix_cos=context["cosine"], prefix_sin=context["sine"], keys=keys, values=values,
                        conditional_output=projected_native, frozen_output=projected_frozen,
                        adaptive_output=projected_adaptive,
                        native_full_attention_output=native_output, native_self_attention_mass=self_mass)
                    if args.save_matrices:
                        arrays["DeltaM"] = np.stack(dense)
                    np.savez_compressed(output/f"context_{context_index:04d}.npz", **arrays)
                    for qi in range(args.queries):
                        query_rows.append(dict(context=context_index, layer=layer, head="all", query=qi,
                            token_id=candidates[qi], is_anchor=qi == 0, prefix_probability=candidate_prob[qi],
                            adaptive_projected_mse=float(np.mean((projected_adaptive[qi]-projected_native[qi])**2)),
                            frozen_projected_mse=float(np.mean((projected_frozen[qi]-projected_native[qi])**2)),
                            constant_output_mse=float(np.mean((projected_native[0]-projected_native[qi])**2))))
                    print(f"Context {context_index+1}/{args.contexts}, layer {layer}: "
                          f"EXACT identity error={block_error:.2e}; "
                          f"adaptive MSE={block_row['adaptive_projected_mse']:.2e}; "
                          f"FROZEN approximation MSE={block_row['frozen_update_transfer_mse']:.6g}; "
                          f"relative RMSE={block_row['frozen_transfer_relative_rmse']}; "
                          f"gauge-specific matrix variation={block_row['literal_update_query_variation']}; native check={native_error:.2e}", flush=True)
                else:
                    block_row.update(status="numerical_validation_failed", error="At least one head failed; no partial-head block statistic was substituted.")
                    print(f"Context {context_index+1}: numerical validation failed; see diagnostics.csv.", flush=True)
                all_rows.extend(context_rows+[block_row])
                metadata["contexts"].append(dict(context=context_index, query_token_ids=candidates,
                    query_probability_mass=float(sum(candidate_prob)),
                    query_hidden_relative_spread=section7a_ratio(float(np.sum((query_hidden-query_hidden.mean(0))**2)),
                                                               float(np.sum(query_hidden**2)), root=True),
                    native_error=native_error, prefix_error=prefix_error, status=block_row["status"]))
                section7a_write_csv(output/"diagnostics.csv", all_rows)
                section7a_write_csv(output/"heads.csv", section7a_summarize(all_rows))
                section7a_write_csv(output/"per_query.csv", query_rows)
                save_metadata()
                capture.clear()
    except Exception as exc:
        metadata.update(status="failed", error=str(exc))
        save_metadata()
        raise
    finally:
        handle.remove()
    metadata["status"] = "completed" if all(r["status"] == "passed" for r in all_rows) else "completed_with_validation_failures"
    save_metadata()
    summary = section7a_summarize(all_rows)
    lines = ["# Pretrained Section 7A shared-update measurement", "",
        f"Checkpoint: {args.model}; layer {layer}; {args.contexts} distinct article prefixes; {args.queries} queries per prefix.", "",
        "The literal identity is checked separately from frozen transfer. Its matrix is query-dependent. The frozen-control error is not an equivalence error. See exact_replay/REPORT.md for exact coefficient refreshing and native full-attention reconstruction.", "",
        "Random smoke model; these are NOT pretrained measurements." if args.smoke else "Frozen pretrained weights; no fitting, gradients, or eligibility gate.", "",
        "Queries are different one-token continuations at the same position. Each reads the same prefix K/V bank; its self entry is excluded. Native full attention is checked separately. This is a conditional attention measurement, not the untouched model's full continuation output.", "",
        "The original-score Section 7A gauge is unchanged. Q/K/V biases are included using [normalized_hidden,1]. RMSNorm is upstream, so these matrices act on normalized attention inputs, not raw residual activations. No derivative or objective is substituted for DeltaM.", "",
        "For each context, variation = ||M_i Wo - mean_i(M_i Wo)||_F / ||M_i Wo||_F, with norms over queries and matrix entries. The all-head row uses the sum of projected matrices including cross-head terms. Linear variation excludes the augmented bias row. Aggregation pools within-context energies, never variation across different contexts.", "",
        "Frozen MSE uses the observed continuation's matrix (query 0) on all other queries. Relative RMSE = sqrt(frozen MSE / constant-anchor-output MSE); below 1 means better transfer than predicting the anchor output for every query. Mean-value baseline MSE is reported too. A zero baseline denominator is undefined, left blank. Contexts, not queries, are the replicate unit; no significance or near-sharing threshold is imposed.", "",
        "| Head | Passed/attempted contexts | Matrix variation | Linear variation | Frozen MSE | Relative RMSE |",
        "|---|---:|---:|---:|---:|---:|"]
    def fmt(value):
        return "undefined" if value is None else f"{value:.6g}"
    for row in summary:
        lines.append(f"| {row['head']} | {row['contexts_passed']}/{row['contexts_attempted']} | "
            +" | ".join(fmt(row.get(k)) for k in ["literal_update_query_variation", "linear_update_query_variation", "frozen_update_transfer_mse", "frozen_transfer_relative_rmse"])+" |")
    lines += ["", "Files: diagnostics.csv contains context/head results and validation failures; heads.csv pools them; per_query.csv excludes no rows but flags query 0; metadata.json records sampling, model revision, configuration, and numerical checks.", "",
        "Exact matrices: context_*.npz stores L[h,m+1,d] and T[h,q,d,d], with DeltaM[h,q] = L[h] @ T[h,q]. weights.npz stores augmented affine projections and Wo[h,d,m]. Add --save-matrices to store dense DeltaM directly. No compression of matrix rank or fitted gauge is used.", "",
        "Small variation and successful transfer would establish approximate sharing only in this gauge, layer, fixed-prefix bank, and sampled query family. They would not establish optimizer dynamics, improved language-model loss, or a RoPE-specific causal effect.", "",
        "Implementation reference: [Qwen2 attention](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen2/modeling_qwen2.py)."]
    (output/"REPORT.md").write_text("\n".join(lines)+"\n")
    print(f"{metadata['status']}: saved diagnostics.csv, heads.csv, per_query.csv, REPORT.md, metadata.json, and exact matrix factors in {output}", flush=True)
    exact_report = section7a_replay(output, output/"exact_replay")
    if exact_report["status"] != "completed":
        raise RuntimeError("Exact replay validation failed; inspect exact_replay/equivalence.csv.")


def select_heads(chi, lag, valid_fraction, mode="none", chi_threshold=CHI_GATE,
                 lag_threshold=MU_GATE, min_valid_fraction=0.25, max_lag=30.0):
    """Separate numerical availability from the optional scientific hypothesis."""
    usable = (np.isfinite(chi) & np.isfinite(lag) &
              np.isfinite(valid_fraction) & (valid_fraction > 0))
    if mode == "none":
        return usable
    if mode == "chi":
        return (usable & (chi >= chi_threshold) & (lag >= lag_threshold) & (lag <= max_lag) &
                (valid_fraction >= min_valid_fraction))
    raise ValueError(f"Unknown gate: {mode}")


def screen_decision(selected_layers, is_last_model):
    if selected_layers:
        return "test_selected_layers"
    return "no_qualifying_heads_in_screened_models" if is_last_model else "screen_next_model"


def screen_and_test(args):
    """Sequential child processes release checkpoint memory between stages."""
    root = Path(args.out); root.mkdir(parents=True, exist_ok=True)
    models = [(args.model, args.revision)]
    if args.secondary_model and args.secondary_model != args.model and not args.smoke:
        models.append((args.secondary_model, args.secondary_revision))
    summary = dict(protocol_version=3, arguments=vars(args), screens=[], experiments=[],
                   status="running", primary_step="uncapped", gate="chi",
                   interpretation="Eligibility is an operational screen on sampled calibration text, not a universal impossibility test.")
    def save():
        (root/"pipeline_summary.json").write_text(json.dumps(summary, indent=2))
    def execute(command, stage):
        print("Running " + stage, flush=True)
        result = subprocess.run(command, check=False)
        if result.returncode:
            summary.update(status="execution_failed", failed_stage=stage,
                           return_code=result.returncode,
                           failure_note="A runtime, access, or numerical failure is not an empty scientific screen.")
            save()
            return False
        return True
    save()
    for model_index, (model_name, revision) in enumerate(models):
        common = [sys.executable, str(Path(__file__).resolve()), "--model", model_name,
            "--revision", revision, "--dataset-revision", args.dataset_revision,
            "--length", str(args.length), "--batch", str(args.batch), "--cal", str(args.cal),
            "--test", str(args.test), "--seed", str(args.seed), "--slope", args.slope,
            "--gate", "chi", "--chi-threshold", str(args.chi_threshold),
            "--lag-threshold", str(args.lag_threshold), "--max-lag", str(args.max_lag),
            "--min-valid-fraction", str(args.min_valid_fraction),
            "--lookahead", str(args.lookahead), "--phase-step", str(args.phase_step),
            "--null-seeds", str(args.null_seeds)]
        if args.smoke:
            common += ["--smoke", "--smoke-arch", args.smoke_arch]
        screen_dir = root/f"model_{model_index}_screen"
        if not execute(common+["--scan-only", "--out", str(screen_dir)], f"all-layer screen: {model_name}"):
            return 1
        meta = json.loads((screen_dir/"metadata.json").read_text())
        layers = meta["selected_layers"]
        summary["screens"].append(dict(model=model_name, selected_layers=layers,
            eligible_heads=meta["eligible_heads"], directory=str(screen_dir), verdict=meta["verdict"]))
        decision = screen_decision(layers, model_index == len(models)-1)
        summary["status"] = decision; save()
        if decision != "test_selected_layers":
            continue
        family = 4*len(layers)
        # Keep about 20 bootstrap samples in each adjusted tail by default.
        bootstrap = max(args.bootstrap, 800*family)
        for layer_index in layers:
            target = root/f"model_{model_index}_layer_{layer_index}"
            command = common+["--layer", str(layer_index), "--uncapped",
                "--contrast-family-size", str(family), "--bootstrap", str(bootstrap),
                "--out", str(target)]
            if not execute(command, f"strict uncapped intervention: {model_name}, layer {layer_index}"):
                return 1
            result = json.loads((target/"metadata.json").read_text())
            summary["experiments"].append(dict(model=model_name, layer=layer_index,
                directory=str(target), selected_alpha=result.get("selected_alpha"),
                verdict=result["verdict"], calibrated_vs_baseline=result.get("calibrated_vs_baseline_exploratory")))
            save()
        summary["status"] = "selected_layers_evaluated"
        save()
        print(f"All selected layers reported in {root/'pipeline_summary.json'}", flush=True)
        return 0
    summary["status"] = "no_qualifying_heads_in_screened_models"
    summary["verdict"] = ("No heads met the stated thresholds in the screened checkpoints and calibration blocks. "
                          "This closes the stated operational branch, not the broader hypothesis.")
    save()
    print(summary["verdict"], flush=True)
    return 0


def articles(lines):
    """WikiText top-level headings; subsections remain inside their article."""
    current = []
    for line in lines:
        if re.fullmatch(r"=\s+[^=]+?\s+=", line.strip()):
            if current:
                yield "\n".join(current)
            current = [line]
        elif current:
            current.append(line)
    if current:
        yield "\n".join(current)


def paired_ci(main, comparator, rng, n_boot, family=1):
    """One block/article. For random controls also resample seed replicates."""
    comparator = np.atleast_2d(comparator)
    n = len(main)
    if n < 2 or comparator.shape[1] != n:
        raise ValueError("Paired inference needs at least two matching articles.")
    boots = np.empty(n_boot)
    for k in range(n_boot):
        ix = rng.integers(n, size=n)
        sx = rng.integers(len(comparator), size=len(comparator))
        boots[k] = main[ix].mean() - comparator[sx][:, ix].mean()
    tail = 0.025 / family
    lo, hi = np.quantile(boots, [tail, 1-tail])
    return dict(delta_nll=float(main.mean()-comparator.mean()),
                ci_lo=float(lo), ci_hi=float(hi), family_size=family)


def self_test():
    # chi=1 can have either sign and arbitrary gain: old correction is wrong.
    x = np.arange(4, dtype=float)
    xc = x-x.mean()
    for gain in [-8.0, -0.8, 0.01, 0.8, 8.0]:
        a = gain*x
        ac = a-a.mean()
        vx, va, gamma = np.mean(xc**2), np.mean(ac**2), np.mean(ac*xc)
        yp = np.mean(xc*ac)
        slope = yp*gamma/(va*vx)
        assert np.isclose(slope, 1)
        assert np.isclose(gamma**2/(va*vx), 1)
    assert not np.isclose((-0.8*1.25)*2.5, 2.5)
    # Independent rotation -> softmax -> readout central finite difference.
    rng = np.random.default_rng(0)
    q, k, v = rng.normal(size=6), rng.normal(size=(9, 6)), rng.normal(size=(9, 5))
    theta = np.tile([1., .1, .01], 2)
    def half(z):
        return np.concatenate([-z[3:], z[:3]])
    def readout(t):
        qt = q*np.cos(t*theta)+half(q)*np.sin(t*theta)
        z = k@qt/math.sqrt(6)
        w = np.exp(z-z.max()); w /= w.sum()
        return w@v, w
    y, w = readout(0)
    a = k@(theta*half(q))/math.sqrt(6)
    yp = (w*(a-w@a))@v
    fd = (readout(1e-5)[0]-readout(-1e-5)[0])/2e-5
    np.testing.assert_allclose(yp, fd, rtol=1e-7, atol=1e-9)
    # Unit-scale increment R2 must retain sign and scale penalties.
    dy = np.array([1., 2., 3.])
    assert 1-np.sum((dy-(-dy))**2)/np.sum(dy**2) == -3
    assert len(list(articles([" = First = ", "text", " = = Sub = = ",
                              "text", " = Second = ", "text"]))) == 2
    ci = paired_ci(np.arange(8.)-1, np.arange(8.), rng, 100, 4)
    assert ci["ci_hi"] < 0
    # Low correlation is not a numerical failure and must not disable the
    # default bounded probe. Strict gating remains strict and may select none.
    ch, lag, valid = np.array([.196, .099, 0.]), np.array([29., 89., .05]), np.array([.36, .98, 0.])
    np.testing.assert_array_equal(select_heads(ch, lag, valid), [True, True, False])
    assert not select_heads(ch, lag, valid, mode="chi").any()
    assert not select_heads(ch, lag, np.zeros(3)).any()
    np.testing.assert_array_equal(select_heads(np.array([.5, .8, .8, .8]),
        np.array([1., 30., 30.01, .99]), np.array([.25]*4), mode="chi"),
        [True, True, False, False])
    assert screen_decision([2], False) == "test_selected_layers"
    assert screen_decision([], False) == "screen_next_model"
    assert screen_decision([], True) == "no_qualifying_heads_in_screened_models"
    print("PASS: projection sign/gain counterexample, independent RoPE finite "
          "difference, signed increment error, article grouping, paired contrast, "
          "optional-gate, lag-boundary, empty-gate, and screening-branch checks.")


def section7a_self_test():
    """Independent affine/relative-RoPE oracle and measurement checks, NumPy only."""
    rng = np.random.default_rng(20260905)
    maxima = dict(matrix=0., output=0., projected_energy=0.)
    for trial in range(32):
        m, d, heads, kv_heads = 7, 4, 4, 2
        n, qcount = (1 if trial == 0 else 9), 6
        context = np.column_stack((rng.normal(size=(n, m)), np.ones(n)))
        query = np.column_stack((rng.normal(size=(qcount, m)), np.ones(qcount)))
        # Deliberately nonzero biases, including a physical zero-input query.
        query[0, :-1] = 0
        wq = rng.normal(size=(heads, m+1, d))*.2
        wk = rng.normal(size=(kv_heads, m+1, d))*.2
        wv = rng.normal(size=(kv_heads, m+1, d))*.3
        wo = rng.normal(size=(heads, d, m))*.4
        frequency = np.array([1., .03, 1., .03])
        positions, qp = np.arange(n)*.7, n+.3
        cosine, sine = np.cos(positions[:, None]*frequency), np.sin(positions[:, None]*frequency)
        qc, qs = np.cos(qp*frequency), np.sin(qp*frequency)
        keys = section7a_rotate(context[None]@wk, cosine[None], sine[None])
        values = context[None]@wv
        block_updates, native_sum, frozen_sum, mu_sum = [], 0., 0., 0.
        for head in range(heads):
            kv = head//(heads//kv_heads)
            result = section7a_effective(query, wq[head], keys[kv], values[kv], qc, qs, d**-.5)
            # Reference from explicit relative rotations and the unscaled expm1
            # identity, independent of the production coefficient implementation.
            slopes = []
            for x, position in zip(context, positions):
                angle = (position-qp)*frequency
                relative = section7a_rotate(np.eye(d), np.cos(angle), np.sin(angle)).T
                slopes.append(wq[head]@relative@wk[kv].T@x/math.sqrt(d))
            slopes = np.stack(slopes, axis=1)
            scores = query@slopes
            rho = np.ones_like(scores)
            np.divide(np.expm1(scores), scores, out=rho, where=scores != 0)
            mu = values[kv].mean(0)
            direct = slopes[None]@((rho/np.exp(scores).sum(1, keepdims=True))[..., None]*(values[kv]-mu)[None])
            np.testing.assert_allclose(result["matrix"], direct, atol=3e-13, rtol=3e-12)
            maxima["matrix"] = max(maxima["matrix"], float(np.max(np.abs(result["matrix"]-direct))))
            maxima["output"] = max(maxima["output"], result["output_error"])
            stats, native, frozen, mean = section7a_head_statistics(result, query, wo[head])
            projected = result["matrix"]@wo[head]
            expected_energy = np.sum(projected**2, axis=(1, 2))
            actual_energy = section7a_energy(result["matrix"], wo[head])
            np.testing.assert_allclose(actual_energy, expected_energy, atol=2e-13, rtol=3e-12)
            maxima["projected_energy"] = max(maxima["projected_energy"], float(np.max(np.abs(expected_energy-actual_energy))))
            expected_transfer = float(np.mean((frozen[1:]-native[1:])**2))
            assert abs(stats["frozen_update_transfer_mse"]-expected_transfer) < 1e-15
            if n == 1:
                assert stats["literal_update_query_variation"] is None
                assert stats["frozen_transfer_relative_rmse"] is None
            block_updates.append(projected)
            native_sum += native
            frozen_sum += frozen
            mu_sum += mean
        block = sum(block_updates)
        # Online all-head variance must retain cross-head terms and equal the
        # original synthetic diagnostic's explicit stacked matrix calculation.
        mean, variance = np.zeros_like(block[0]), 0.
        for qi, update in enumerate(block):
            delta = update-mean
            mean += delta/(qi+1)
            variance += float(np.sum(delta*(update-mean)))
        np.testing.assert_allclose(variance, np.sum((block-block.mean(0))**2), atol=5e-13, rtol=3e-12)
        transfer = section7a_transfer(native_sum, frozen_sum, mu_sum)
        assert abs(transfer["frozen_update_transfer_mse"]-np.mean((frozen_sum[1:]-native_sum[1:])**2)) < 1e-15
    # Exact zero scores, mixed extreme scores, and a literal-gauge range failure.
    for scores in [np.zeros((2, 5)), np.array([[0., 1e-12, -.099, .1, 700., -700.]])]:
        coef = section7a_coefficients(scores)
        shift = scores.max(1, keepdims=True)
        p = np.exp(scores-shift); p /= p.sum(1, keepdims=True)
        v = rng.normal(size=scores.shape[1]); vc = v-v.mean()
        np.testing.assert_allclose(np.sum(scores*coef*vc, axis=1)+v.mean(), p@v, atol=2e-13, rtol=1e-12)
    try:
        section7a_coefficients(np.full((1, 4), -1000.))
    except FloatingPointError:
        pass
    else:
        raise AssertionError("An unrepresentable literal gauge was silently accepted.")
    # Pool within-context energies; do not center matrices across contexts or
    # average their ratios. Two unequal-energy contexts distinguish the rules.
    rows = []
    for energy, variance, error in [(1., .25, 1.), (9., 1., 3.)]:
        rows.append(dict(head=0, status="passed", projected_matrix_energy=energy,
            projected_matrix_variation_energy=variance, linear_matrix_energy=energy,
            linear_matrix_variation_energy=variance, literal_update_query_variation=math.sqrt(variance/energy),
            frozen_update_transfer_mse=error, constant_output_baseline_mse=4., mean_value_baseline_mse=5., output_max_abs_error=0.))
    pooled = section7a_summarize(rows)[0]
    np.testing.assert_allclose(pooled["literal_update_query_variation"], math.sqrt(1.25/10.))
    np.testing.assert_allclose(pooled["frozen_transfer_relative_rmse"], math.sqrt(.5))
    print("PASS: Section 7A affine biases, split-half relative rotations, grouped heads, "
          "literal matrices, zero/extreme scores, projected Frobenius norms, cross-head "
          "variance, frozen transfer, degenerate baselines, and context pooling. "
          +"; ".join(f"max {k} error={v:.2e}" for k, v in maxima.items()))


def section7a_refresh_self_test():
    """Test missing-term recovery, full self attention, and the complete replay CLI data path."""
    import contextlib
    import io
    rng = np.random.default_rng(5123)
    max_refresh = max_full = max_frozen = 0.
    # A nonlinear counterexample: exact matrices vary while frozen transfer fails.
    for trial in range(16):
        qn, n, m, d = 5, 7, 6, 4
        query = np.column_stack((rng.normal(size=(qn, m))*2, np.ones(qn)))
        left = rng.normal(size=(m+1, d))*.4
        keys, values = rng.normal(size=(n, d)), rng.normal(size=(n, d))
        wk, wv = rng.normal(size=(m+1, d))*.3, rng.normal(size=(m+1, d))*.5
        phase = np.tile(np.array([1.2, .09]), 2)
        cs, sn = np.cos(phase), np.sin(phase)
        prefix = section7a_refresh(query, left, keys, values, anchor=trial % qn)
        full = section7a_full_readout(query, left, keys, values, wk, wv, cs, sn)
        max_refresh = max(max_refresh, float(np.max(np.abs(prefix["exact"]-prefix["direct"]))))
        max_full = max(max_full, float(np.max(np.abs(full["literal"]-full["direct"]))))
        max_frozen = max(max_frozen, float(np.max(np.abs(prefix["frozen"]-prefix["direct"]))))
        # Direct independent per-query matrix sum, including a changing self K/V
        # and unweighted full-bank mean. No probability mixing used in this oracle.
        for qi in range(qn):
            kself = section7a_rotate(query[qi]@wk, cs, sn)
            vself = query[qi]@wv
            all_keys = np.vstack((keys, kself))
            all_values = np.vstack((values, vself))
            slopes = left@all_keys.T
            scores = query[qi]@slopes
            rho = np.ones_like(scores)
            np.divide(np.expm1(scores), scores, out=rho, where=scores != 0)
            mean = all_values.mean(0)
            matrix = slopes@((rho/np.exp(scores).sum())[:, None]*(all_values-mean))
            np.testing.assert_allclose(left@full["fast"][qi], matrix, atol=2e-11, rtol=2e-10)
            np.testing.assert_allclose(full["mu"][qi], mean, atol=1e-14, rtol=1e-13)
    assert max_frozen > .1, "The fixture must exhibit a real frozen approximation error."
    # Generate a version-4-format recording, then run the SAME saved-factor replay
    # used for pretrained captures, including output files and failure detection.
    H, HK, n, qn, m, d = 4, 2, 9, 5, 6, 4
    wq = rng.normal(size=(H, m+1, d))*.3
    wk = rng.normal(size=(HK, m+1, d))*.3
    wv = rng.normal(size=(HK, m+1, d))*.4
    wo = rng.normal(size=(H, d, m))*.3
    query = np.column_stack((rng.normal(size=(qn, m))*2, np.ones(qn)))
    context = np.column_stack((rng.normal(size=(n, m)), np.ones(n)))
    frequency = np.array([1., .02, 1., .02])
    cp, qp = np.arange(n), float(n)
    cs, sn = np.cos(cp[:, None]*frequency), np.sin(cp[:, None]*frequency)
    qc, qs = np.cos(qp*frequency), np.sin(qp*frequency)
    keys = section7a_rotate(context[None]@wk, cs[None], sn[None])
    values = context[None]@wv
    exact, frozen, native = [np.zeros((qn, m)) for _ in range(3)]
    lefts, fasts, means, masses, old_rows, updates = [], [], [], [], [], []
    for head in range(H):
        kv = head//(H//HK)
        original = section7a_effective(query, wq[head], keys[kv], values[kv], qc, qs, d**-.5)
        stats, yy, ff, _ = section7a_head_statistics(original, query, wo[head])
        exact += yy; frozen += ff
        old_rows.append(dict(context=0, head=head, **stats))
        lefts.append(original["left"]); fasts.append(original["fast"]); means.append(original["mu"])
        updates.append(original["matrix"]@wo[head])
        # Independent native attention oracle, not the new full-readout function.
        self_keys = section7a_rotate(query@wk[kv], qc, qs)
        self_values = query@wv[kv]
        qrot = section7a_rotate(query@wq[head], qc, qs)
        logits = np.column_stack((qrot@keys[kv].T, np.einsum("qd,qd->q", qrot, self_keys)))/math.sqrt(d)
        probability = np.exp(logits-logits.max(1, keepdims=True)); probability /= probability.sum(1, keepdims=True)
        native += (probability[:, :-1]@values[kv]+probability[:, -1:]*self_values)@wo[head]
        masses.append(probability[:, -1])
    update = sum(updates)
    old_rows.append(dict(context=0, head="all", frozen_update_transfer_mse=float(np.mean((frozen[1:]-exact[1:])**2)),
        literal_update_query_variation=float(np.linalg.norm(update-update.mean(0))/np.linalg.norm(update))))
    with tempfile.TemporaryDirectory(prefix="section7a_replay_test_", dir=Path(__file__).resolve().parent) as temporary:
        root = Path(temporary); source = root/"recording"; source.mkdir()
        np.savez_compressed(source/"weights.npz", Wq_aug=wq, Wk_aug=wk, Wv_aug=wv, Wo=wo,
                            head_to_kv=np.arange(H)//(H//HK))
        data = dict(query_aug=query, prefix_hidden_aug=context, query_cos=qc, query_sin=qs, prefix_cos=cs, prefix_sin=sn,
            L=np.stack(lefts), T=np.stack(fasts), mu=np.stack(means), keys=keys, values=values,
            native_self_attention_mass=np.stack(masses, axis=1), conditional_output=exact, frozen_output=frozen,
            native_full_attention_output=native.astype(np.float32), query_token_ids=np.arange(qn))
        np.savez_compressed(source/"context_0000.npz", **data)
        (source/"metadata.json").write_text(json.dumps(dict(protocol_version=4, model="synthetic_fixture", layer=0,
                                                           checkpoint_kind="synthetic_fixture")))
        section7a_write_csv(source/"diagnostics.csv", old_rows)
        with contextlib.redirect_stdout(io.StringIO()):
            replay = section7a_replay(source, root/"valid")
        assert replay["status"] == "completed"
        assert len(replay["summary"]) == H+1
        whole = replay["summary"][-1]
        assert whole["prefix_anchor_plus_refresh_mse"] < 1e-22
        assert whole["full_literal_mse"] < 1e-22
        np.testing.assert_allclose(whole["prefix_frozen_mse"], old_rows[-1]["frozen_update_transfer_mse"], atol=1e-12, rtol=1e-11)
        for name in ["REPORT.md", "metadata.json", "heads.csv", "equivalence.csv", "per_query.csv", "replay_0000.npz"]:
            assert (root/"valid"/name).stat().st_size > 0
        # A corrupted factor must produce an explicit failure, never a passing
        # report that silently omits the bad head or overwrites the old control.
        data["T"] = data["T"].copy(); data["T"][0, 0, 0, 0] += .125
        np.savez_compressed(source/"context_0000.npz", **data)
        with contextlib.redirect_stdout(io.StringIO()):
            bad = section7a_replay(source, root/"corrupt")
        assert bad["status"] == "completed_with_validation_failures"
        assert bad["summary"][-1]["contexts_passed"] == 0
    print(f"PASS: query refresh and full self-K/V literal matrices; max refresh error={max_refresh:.2e}, "
          f"max full error={max_full:.2e}, counterexample frozen error={max_frozen:.3g}; "
          "version-4 replay, original-control preservation, report generation, and corrupted-factor rejection.")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--self-test", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--smoke-arch", choices=["qwen2", "qwen3", "llama"], default="qwen2")
    stages = ap.add_mutually_exclusive_group()
    stages.add_argument("--scan-only", action="store_true", help="all layers, calibration only, no gradients or interventions")
    stages.add_argument("--screen-and-test", action="store_true", help="screen Qwen, test qualifying layers uncapped; screen Llama if Qwen has none")
    stages.add_argument("--shared-update", action="store_true", help="literal Section 7A fixed-prefix per-head query variation and frozen transfer")
    stages.add_argument("--replay", metavar="RESULTS_DIR", help="NumPy-only exact query refresh and native full-attention checks from saved weights/context factors")
    ap.add_argument("--contexts", type=int, default=16, help="distinct article prefixes for --shared-update")
    ap.add_argument("--queries", type=int, default=32, help="same-position continuation queries per prefix for --shared-update")
    ap.add_argument("--save-matrices", action="store_true", help="also store dense DeltaM[h,q,m+1,d]; exact L,T factors are always saved")
    ap.add_argument("--secondary-model", default="meta-llama/Llama-3.2-1B")
    ap.add_argument("--secondary-revision", default="main")
    ap.add_argument("--model", default=MODEL_ID)
    ap.add_argument("--revision", default="main")
    ap.add_argument("--dataset-revision", default="main")
    ap.add_argument("--layer", type=int, default=-1)
    ap.add_argument("--length", type=int, default=SEQ_LEN)
    ap.add_argument("--batch", type=int, default=BATCH)
    ap.add_argument("--cal", type=int, default=N_CAL_BLOCKS)
    ap.add_argument("--test", type=int, default=N_TEST_BLOCKS)
    ap.add_argument("--slope", choices=["projected", "phase", "regress", "momentum"], default=SLOPE)
    ap.add_argument("--gate", choices=["none", "chi"], default="none",
                    help="none: probe all numerically usable heads; chi: impose the optional correlation hypothesis")
    ap.add_argument("--chi-threshold", type=float, default=CHI_GATE)
    ap.add_argument("--lag-threshold", type=float, default=MU_GATE)
    ap.add_argument("--max-lag", type=float, default=30.0)
    ap.add_argument("--min-valid-fraction", type=float, default=0.25,
                    help="minimum valid calibration fraction for --gate chi only")
    ap.add_argument("--lookahead", type=float, default=LOOKAHEAD)
    ap.add_argument("--phase-step", type=float, default=PHASE_STEP)
    ap.add_argument("--cap", type=float, default=MAX_REL_UPDATE)
    ap.add_argument("--uncapped", action="store_true", help="retain raw correction magnitude; no stability or loss guarantee is asserted")
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--bootstrap", type=int, default=N_BOOT)
    ap.add_argument("--null-seeds", type=int, default=NULL_SEEDS)
    ap.add_argument("--contrast-family-size", type=int, default=4)
    ap.add_argument("--out", default="results_v3")
    args = ap.parse_args()
    if args.self_test:
        self_test()
        section7a_self_test()
        section7a_refresh_self_test()
        return
    if args.replay:
        try:
            result = section7a_replay(args.replay, args.out)
        except (FileNotFoundError, FileExistsError, ValueError) as exc:
            raise SystemExit(str(exc))
        if result["status"] != "completed":
            raise SystemExit(1)
        return
    if args.shared_update:
        if args.contexts < 1 or args.queries < 2 or args.batch < 1 or args.length < 2:
            ap.error("--shared-update needs contexts >= 1, queries >= 2, batch >= 1, and length >= 2.")
        if Path(args.out).exists() and any(Path(args.out).iterdir()):
            ap.error("--shared-update requires a new or empty --out directory.")
    if args.max_lag < args.lag_threshold or args.contrast_family_size < 4:
        raise SystemExit("Need max-lag >= lag-threshold and contrast-family-size >= 4.")
    if args.screen_and_test:
        raise SystemExit(screen_and_test(args))
    try:
        import torch
        import torch.nn.functional as F
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as e:
        raise SystemExit("Install dependencies using the command in this file's docstring. " + str(e))
    if args.shared_update and not args.smoke:
        print("Preflight: checking the native Qwen hook and complete Section 7A report on a tiny random model.", flush=True)
        with tempfile.TemporaryDirectory(prefix="section7a_preflight_") as temporary:
            checked = subprocess.run([sys.executable, str(Path(__file__).resolve()),
                "--shared-update", "--smoke", "--contexts", "1", "--queries", "3",
                "--batch", "2", "--out", str(Path(temporary)/"results")],
                text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            if checked.returncode:
                raise SystemExit("Qwen preflight failed before checkpoint loading:\n"+checked.stdout)
            preflight = json.loads((Path(temporary)/"results"/"metadata.json").read_text())
            if preflight["status"] != "completed":
                raise SystemExit("Qwen preflight failed numerical validation:\n"+checked.stdout)
            args.shared_preflight = dict(status="passed", output=checked.stdout)
        print("Preflight passed; loading the pretrained checkpoint.", flush=True)
    if args.smoke:
        args.length, args.cal, args.test, args.bootstrap = 24, 4, 6, 200
        args.contexts, args.queries = min(args.contexts, 2), min(args.queries, 4)
    if min(args.length, args.batch, args.cal, args.test) < 2 or args.cap <= 0:
        # Batch 1 is supported, including an uneven final batch.
        if not (args.length >= 2 and args.batch >= 1 and args.cal >= 2 and args.test >= 2 and args.cap > 0):
            raise SystemExit("Need length/cal/test >= 2, batch >= 1, and positive cap.")
    if args.null_seeds < 2 or args.bootstrap < 100:
        raise SystemExit("Need at least two null seeds and 100 bootstrap replicates.")
    if not 0 <= args.chi_threshold <= 1 or not 0 <= args.min_valid_fraction <= 1:
        raise SystemExit("Correlation and valid-fraction thresholds must lie in [0, 1].")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device("cpu" if args.smoke or not torch.cuda.is_available() else "cuda")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = False
    if args.smoke:
        from transformers import Qwen2Config, Qwen2ForCausalLM
        Cfg, Mdl = Qwen2Config, Qwen2ForCausalLM
        if args.smoke_arch == "qwen3":
            from transformers import Qwen3Config as Cfg, Qwen3ForCausalLM as Mdl
        elif args.smoke_arch == "llama":
            from transformers import LlamaConfig as Cfg, LlamaForCausalLM as Mdl
        cfg = Cfg(vocab_size=128, hidden_size=64, intermediate_size=128,
                  num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                  head_dim=16, max_position_embeddings=256, rope_theta=1000.)
        cfg._attn_implementation = "eager"
        model, tok = Mdl(cfg), None
    else:
        tok = AutoTokenizer.from_pretrained(args.model, revision=args.revision)
        version = importlib.metadata.version("transformers").split(".")
        modern_dtype = (int(version[0]), int(version[1])) >= (4, 56)
        dtype_kw = {"dtype" if modern_dtype else "torch_dtype": torch.float32}
        model = AutoModelForCausalLM.from_pretrained(args.model, revision=args.revision,
                    attn_implementation="eager", **dtype_kw)
    model = model.to(device).float().eval()
    model.config.use_cache = False
    for p in model.parameters():
        p.requires_grad_(False)
    cfg, core = model.config, model.model
    if cfg.model_type not in {"qwen2", "qwen3", "llama"}:
        raise SystemExit("Supported architectures: Qwen2, Qwen3, Llama, full fixed-frequency RoPE.")
    if getattr(cfg, "use_sliding_window", False) or "sliding_attention" in (getattr(cfg, "layer_types", None) or []):
        raise SystemExit("Sliding attention is unsupported.")
    for spec in [getattr(cfg, "rope_scaling", None), getattr(cfg, "rope_parameters", None)]:
        if spec and spec.get("rope_type", spec.get("type", "default")) not in {"default", "llama3"}:
            raise SystemExit("Only default and static llama3 frequency schedules are supported.")
    li = args.layer if args.layer >= 0 else len(core.layers)+args.layer
    if not 0 <= li < len(core.layers):
        raise SystemExit("Selected layer is outside the model.")
    module = core.layers[li].self_attn
    H = cfg.num_attention_heads
    HK = getattr(cfg, "num_key_value_heads", None) or H
    D = getattr(cfg, "head_dim", None) or cfg.hidden_size//H
    rotary = getattr(core, "rotary_emb", None)
    if rotary is None:
        rotary = getattr(module, "rotary_emb", None)
    if rotary is None or 2*rotary.inv_freq.numel() != D or H % HK:
        raise SystemExit("Unsupported rotary layout or GQA dimensions.")
    theta = rotary.inv_freq.detach().float().to(device).repeat(2)
    output = Path(args.out); output.mkdir(parents=True, exist_ok=True)
    data_info = {}

    def load_blocks(split, n):
        if args.smoke:
            gen = torch.Generator().manual_seed(args.seed + (0 if split == "train" else 1))
            ids = torch.randint(0, 128, (n, args.length), generator=gen)
            data_info[split] = [{"article": i, "offset": 0} for i in range(n)]
            return ids
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=split,
                          revision=args.dataset_revision)
        docs = list(articles(ds["text"]))
        rng = np.random.default_rng(args.seed + (0 if split == "train" else 1))
        blocks, info = [], []
        for ix in rng.permutation(len(docs)):
            ids = tok(docs[ix], add_special_tokens=False)["input_ids"]
            if len(ids) < args.length:
                continue
            start = int(rng.integers(len(ids)-args.length+1))
            block = ids[start:start+args.length]
            blocks.append(block)
            info.append(dict(article=int(ix), offset=start,
                             token_sha256=hashlib.sha256(np.array(block, dtype="<i8").tobytes()).hexdigest()))
            if len(blocks) == n:
                break
        if len(blocks) < n:
            raise SystemExit(f"{split}: only {len(blocks)} eligible distinct articles; lower --cal/--test or --length.")
        data_info[split] = {"fingerprint": ds._fingerprint, "blocks": info}
        return torch.tensor(blocks, dtype=torch.long)

    if args.shared_update:
        shared_update_probe(args, model, tok, module, rotary, li, load_blocks, data_info, output)
        return

    cal = load_blocks("train", args.cal)
    state = dict(enabled=False, alpha=torch.zeros(H, device=device), mode="main",
                 seed=args.seed, start=0, record=False, checked=False, stats=[], verify=False,
                 screening=False, screen_layer=None, screen_stats={})

    def half(t):
        return torch.cat([-t[..., D//2:], t[..., :D//2]], dim=-1)

    def norm_match(candidate, base):
        cn = candidate.norm(dim=-1, keepdim=True)
        # Degenerate permutations fall back to a nonzero coordinate sign control.
        fallback = -base
        candidate = torch.where(cn > 1e-20, candidate, fallback)
        return candidate * (base.norm(dim=-1, keepdim=True) /
                            candidate.norm(dim=-1, keepdim=True).clamp_min(1e-20))

    def hook(mod, positional, kw, native):
        if not state["enabled"]:
            return native
        ref = native[0] if isinstance(native, tuple) else native
        hs = kw.get("hidden_states")
        if hs is None:
            hs = positional[0]
        b, length, _ = hs.shape
        with torch.no_grad():
            pe = kw.get("position_embeddings")
            if pe is None:
                pe = rotary(hs, torch.arange(length, device=device)[None])
            cs, sn = pe
            expected = torch.arange(length, device=device)[:, None]*theta[None]
            if state["verify"]:
                torch.testing.assert_close(cs.float(), expected.cos()[None].expand_as(cs), atol=2e-5, rtol=2e-5)
                torch.testing.assert_close(sn.float(), expected.sin()[None].expand_as(sn), atol=2e-5, rtol=2e-5)
            cs, sn = cs[:, None], sn[:, None]
            q = mod.q_proj(hs).view(b, length, H, D)
            k = mod.k_proj(hs).view(b, length, HK, D)
            v = mod.v_proj(hs).view(b, length, HK, D).transpose(1, 2)
            if hasattr(mod, "q_norm"):
                q = mod.q_norm(q)
            if hasattr(mod, "k_norm"):
                k = mod.k_norm(k)
            q, k = q.transpose(1, 2), k.transpose(1, 2)
            q, k = q*cs+half(q)*sn, k*cs+half(k)*sn
            k, v = k.repeat_interleave(H//HK, 1), v.repeat_interleave(H//HK, 1)
            scale = getattr(mod, "scaling", D**-0.5)
            z = (q@k.transpose(-1, -2))*scale
            causal = torch.ones(length, length, device=device, dtype=torch.bool).tril()
            z = z.masked_fill(~causal, float("-inf"))
            mask = kw.get("attention_mask")
            if mask is not None:
                if mask.dtype == torch.bool or mask.ndim != 4:
                    raise RuntimeError("Expected an additive 4-D causal mask.")
                z = z + mask[..., :length, :length]
            w = torch.softmax(z, -1)
            y = w@v
            if state["verify"]:
                recomputed = mod.o_proj(y.transpose(1, 2).reshape(b, length, H*D))
                torch.testing.assert_close(recomputed, ref, atol=2e-5, rtol=2e-4)
            a = ((theta*half(q))@k.transpose(-1, -2))*scale
            pos = torch.arange(length, device=device, dtype=torch.float32)
            x = pos[None, :]-pos[:, None]
            mx = (w*x).sum(-1, keepdim=True)
            xc = x-mx
            ac = a-(w*a).sum(-1, keepdim=True)
            vx = (w*xc.square()).sum(-1, keepdim=True)
            va = (w*ac.square()).sum(-1, keepdim=True)
            gamma = (w*ac*xc).sum(-1, keepdim=True)
            effective = 1/w.square().sum(-1, keepdim=True)
            ok = (vx > 1e-6) & (va > 1e-12) & (effective >= 3)
            denom = (va*vx).clamp_min(1e-30)
            chi = torch.where(ok, gamma.square()/denom, torch.zeros_like(gamma))
            if not torch.isfinite(chi).all() or not torch.isfinite(mx).all():
                raise RuntimeError("Non-finite attention diagnostics; cannot interpret the screen.")
            if chi.max().item() > 1.001:
                raise RuntimeError("Invalid correlation: numerical instability.")
            chi = chi.clamp(0, 1)
            if state["screening"]:
                # Record only and return the original object. No correction is
                # constructed; no output projection, backward pass, or logits.
                reduce = lambda t: t[:, :, :-1, 0].double().sum((0, 2)).cpu().numpy()
                screen_stats = dict(chi=reduce(chi), count=reduce(ok), lag=reduce(-mx),
                    rows=b*(length-1), sigma=reduce(vx.sqrt()),
                    effective=reduce(effective), support_ok=reduce(effective >= 3))
                state["screen_stats"][state["screen_layer"]].append(screen_stats)
                return native
            yp = (w*ac)@v
            factor = torch.where(ok, gamma/denom, torch.zeros_like(gamma))
            if args.slope == "projected":
                raw = (-mx+args.lookahead)*(yp*factor)
            elif args.slope == "regress":
                raw = (-mx+args.lookahead)*((w*xc)@v)/vx.clamp_min(1e-6)
            elif args.slope == "phase":
                raw = args.phase_step*yp
            else:
                raw = torch.cat([torch.zeros_like(y[:, :, :1]), y[:, :, 1:]-y[:, :, :-1]], 2)
            raw = torch.where(ok, raw, torch.zeros_like(raw))
            raw[:, :, -1] = 0  # last query has no scored next-token target
            cap = args.cap*y.norm(dim=-1, keepdim=True)
            would_clip = raw.norm(dim=-1)>cap[..., 0]
            d = raw.clone() if args.uncapped else raw * (cap/raw.norm(dim=-1, keepdim=True).clamp_min(1e-20)).clamp_max(1)
            base = d.clone()
            if state["mode"] != "main":
                for bi in range(b):
                    gen = torch.Generator(device=device).manual_seed(state["seed"]+1000003*(state["start"]+bi))
                    if state["mode"] == "random":
                        candidate = torch.randn(d[bi].shape, device=device, generator=gen)
                    elif args.slope == "momentum":
                        ix = torch.rand(d[bi].shape, device=device, generator=gen).argsort(-1)
                        candidate = d[bi].gather(-1, ix)
                    else:
                        ix = torch.rand((H, length, length), device=device, generator=gen).masked_fill(~causal, 2).argsort(-1)
                        covariate = xc.expand(b, H, length, length)[bi] if args.slope == "regress" else ac[bi]
                        cp = covariate.gather(-1, ix)
                        cp = cp-(w[bi]*cp).sum(-1, keepdim=True)
                        candidate = (w[bi]*cp)@v[bi]
                        if args.slope == "projected":
                            candidate = candidate*factor[bi]*(-mx[bi]+args.lookahead)
                        elif args.slope == "phase":
                            candidate = candidate*args.phase_step
                        else:
                            candidate = candidate*(-mx[bi]+args.lookahead)
                    d[bi] = norm_match(candidate, base[bi])
                torch.testing.assert_close(d.norm(dim=-1), base.norm(dim=-1), atol=1e-7, rtol=1e-5)
            if not torch.isfinite(d).all():
                raise RuntimeError("Non-finite intervention.")
            if state["verify"] and not state["checked"]:
                # Independent FLOAT64 finite difference of the actual fixed-memory readout.
                qr, kr, vv = q[0, 0, -2].double(), k[0, 0, :-1].double(), v[0, 0, :-1].double()
                th = theta.double()
                def local_readout(t):
                    qt = qr*(t*th).cos()+half(qr)*(t*th).sin()
                    return torch.softmax((kr@qt)*scale, -1)@vv
                ww = torch.softmax((kr@qr)*scale, -1)
                aa = (kr@(th*half(qr)))*scale
                analytic = (ww*(aa-(ww*aa).sum()))@vv
                for step in [1e-3, 1e-4]:
                    fd = (local_readout(step)-local_readout(-step))/(2*step)
                    torch.testing.assert_close(analytic, fd, rtol=2e-4, atol=1e-7)
                state["checked"] = True
            if state["record"]:
                sl = slice(None, -1)
                # Real sequence increments change content AND memory; predictive test only.
                dy, pred = y[:, :, 1:]-y[:, :, :-1], yp[:, :, :-1]
                eligible = ok[:, :, :-1]
                dy, pred = dy*eligible, pred*eligible
                reduce = lambda t: t.double().sum((0, 2)).cpu().numpy()
                state["stats"].append(dict(
                    chi=reduce(chi[:, :, sl, 0]), count=reduce(ok[:, :, sl, 0]),
                    lag=reduce(-mx[:, :, sl, 0]), rows=b*(length-1),
                    xy=reduce((dy*pred).sum(-1)), xx=reduce(pred.square().sum(-1)),
                    yy=reduce(dy.square().sum(-1)),
                    clipped=reduce(would_clip[:, :, sl] if not args.uncapped else torch.zeros_like(would_clip[:, :, sl])),
                    would_clip=reduce(would_clip[:, :, sl]),
                    ratio_sum=reduce(raw[:, :, sl].norm(dim=-1)/y[:, :, sl].norm(dim=-1).clamp_min(1e-20)),
                ))
        # Add only the intervention to native attention output. Never add bias twice.
        change = (d*state["alpha"].view(1, H, 1, 1)).transpose(1, 2).reshape(b, length, H*D)
        new = ref+F.linear(change, mod.o_proj.weight, bias=None)
        return (new,)+native[1:] if isinstance(native, tuple) else new

    if args.scan_only:
        state.update(enabled=True, screening=True, screen_stats={i: [] for i in range(len(core.layers))})
        def screen_hook(layer_index):
            def callback(mod, positional, kw, native):
                state["screen_layer"] = layer_index
                recorded = hook(mod, positional, kw, native)
                if recorded is not native:
                    raise RuntimeError("Screen must return the exact native attention output object.")
                return native
            return callback
        handles = [layer.self_attn.register_forward_hook(screen_hook(i), with_kwargs=True)
                   for i, layer in enumerate(core.layers)]
        forward_calls = 0
        try:
            with torch.no_grad():
                for start in range(0, len(cal), args.batch):
                    state["verify"] = start == 0
                    core(input_ids=cal[start:start+args.batch].to(device), use_cache=False)
                    forward_calls += 1
                    print(f"Screened {min(start+args.batch, len(cal))}/{len(cal)} calibration blocks across {len(core.layers)} layers.", flush=True)
        finally:
            for registered in handles:
                registered.remove()
        rows, per_batch = [], []
        for layer_index, records in state["screen_stats"].items():
            if len(records) != forward_calls:
                raise RuntimeError(f"Missing scan records for layer {layer_index}.")
            st = {key: sum(item[key] for item in records) for key in records[0]}
            ch = st["chi"]/np.maximum(st["count"], 1)
            lag, valid = st["lag"]/st["rows"], st["count"]/st["rows"]
            selected = select_heads(ch, lag, valid, "chi", args.chi_threshold,
                                    args.lag_threshold, args.min_valid_fraction, args.max_lag)
            for h in range(H):
                rows.append(dict(layer=layer_index, head=h, chi=float(ch[h]),
                    mean_lag=float(lag[h]), valid_fraction=float(valid[h]), eligible=bool(selected[h]),
                    mean_position_sd=float(st["sigma"][h]/st["rows"]),
                    mean_effective_support=float(st["effective"][h]/st["rows"]),
                    fraction_support_ge3=float(st["support_ok"][h]/st["rows"])))
            for batch_index, item in enumerate(records):
                for h in range(H):
                    per_batch.append(dict(layer=layer_index, head=h, batch=batch_index,
                        chi=float(item["chi"][h]/max(item["count"][h], 1)),
                        mean_lag=float(item["lag"][h]/item["rows"]),
                        valid_fraction=float(item["count"][h]/item["rows"])))
        eligible = [row for row in rows if row["eligible"]]
        selected_layers = sorted({row["layer"] for row in eligible})
        write_csv(output/"all_layers_heads.csv", rows)
        write_csv(output/"eligible_heads.csv", eligible, fieldnames=list(rows[0]))
        write_csv(output/"screen_batches.csv", per_batch)
        verdict = (f"{len(eligible)} heads meet the stated calibration criteria in {len(selected_layers)} layers."
                   if eligible else "No heads meet the stated criteria in these calibration blocks; no qualifying substrate under this operational definition.")
        meta = dict(protocol_version=3, stage="all_layer_screen", arguments=vars(args),
            model_commit=getattr(cfg, "_commit_hash", None), model_config=cfg.to_dict(), data=data_info,
            layers_scanned=len(core.layers), heads_scanned=len(rows), eligible_heads=eligible,
            selected_layers=selected_layers, native_forward_calls=forward_calls,
            blocks_evaluated=len(cal), gradient_passes=0, interventions=0, heldout_evaluated=False,
            baseline_check="Every hook returned the original native output object; recomputation and frequency checks ran on the first batch in every layer.",
            chi_definition="Mean of squared score-position correlation over numerically valid rows; final unscored query excluded.",
            validity_definition="Var(position)>1e-6, Var(score derivative)>1e-12, effective attention support>=3.",
            interpretation="Large mean lag does not by itself establish content averaging; gate failure does not prove no positional mechanism.",
            source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
            sources=["https://github.com/huggingface/transformers/blob/main/src/transformers/modeling_rope_utils.py",
                     "https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/convert_llama_weights_to_hf.py"],
            verdict=verdict)
        (output/"metadata.json").write_text(json.dumps(meta, indent=2, default=str))
        print(verdict, flush=True)
        for row in eligible:
            print(f"layer={row['layer']:2d} head={row['head']:2d} chi={row['chi']:.4f} lag={row['mean_lag']:.2f} valid={row['valid_fraction']:.3f}", flush=True)
        return

    handle = module.register_forward_hook(hook, with_kwargs=True)

    def losses(blocks, alpha=None, mode="main", seed=None, grad=False, record=False):
        state.update(enabled=alpha is not None, alpha=alpha, mode=mode,
                     seed=args.seed if seed is None else seed, record=record, stats=[])
        values = []
        context = torch.enable_grad if grad else torch.no_grad
        with context():
            for start in range(0, len(blocks), args.batch):
                state["start"] = start
                ids = blocks[start:start+args.batch].to(device)
                logits = model(input_ids=ids, use_cache=False).logits[:, :-1].float()
                nll = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="none").view(len(ids), -1)
                if not torch.isfinite(nll).all():
                    raise RuntimeError("Non-finite loss.")
                if grad:
                    (nll.sum()/(len(blocks)*(args.length-1))).backward()
                values.extend(nll.detach().double().mean(1).cpu().tolist())
        return np.asarray(values)

    def aggregate():
        return {k: sum(s[k] for s in state["stats"]) for k in state["stats"][0]}

    try:
        zero = torch.zeros(H, device=device)
        native = losses(cal[:args.batch])
        state["verify"] = True
        hooked_zero = losses(cal[:args.batch], zero)
        np.testing.assert_array_equal(native, hooked_zero)
        state["verify"] = False
        alpha = torch.zeros(H, device=device, requires_grad=True)
        cal_base = losses(cal, alpha, grad=True, record=True)
        st = aggregate()
        gradient = alpha.grad.detach().cpu().numpy()
        chi = st["chi"]/np.maximum(st["count"], 1)
        lag = st["lag"]/st["rows"]
        valid_fraction = st["count"]/st["rows"]
        gate = select_heads(chi, lag, valid_fraction, args.gate,
                            args.chi_threshold, args.lag_threshold, args.min_valid_fraction, args.max_lag)
        strict_gate = select_heads(chi, lag, valid_fraction, "chi",
                                   args.chi_threshold, args.lag_threshold, args.min_valid_fraction, args.max_lag)
        gate_t = torch.tensor(gate, device=device, dtype=torch.float32)
        print(f"Layer {li}: {gate.sum()}/{H} heads active (--gate {args.gate}); "
              f"strict chi gate would select {strict_gate.sum()}/{H}; "
              f"max chi={chi.max():.4f}; native baseline preserved.", flush=True)
        calibration_rows = [dict(layer=li, head=h, gradient=float(gradient[h]),
            chi=float(chi[h]), mean_lag=float(lag[h]), valid_fraction=float(valid_fraction[h]),
            active=bool(gate[h]), strict_chi_eligible=bool(strict_gate[h]),
            clipped_fraction=float(st["clipped"][h]/st["rows"]),
            would_clip_at_cap_fraction=float(st["would_clip"][h]/st["rows"]),
            mean_raw_update_ratio=float(st["ratio_sum"][h]/st["rows"])) for h in range(H)]
        write_csv(output/"heads_calibration.csv", calibration_rows)
        if not gate.any():
            verdict = ("STOPPED BEFORE HELD-OUT EVALUATION: no heads selected. "
                       "No intervention comparison was run; no efficacy conclusion is possible.")
            metadata = dict(protocol_version=3, arguments=vars(args), selected_layer=li,
                stage="calibration_only", heldout_evaluated=False, active_heads=0,
                max_chi=float(chi.max()), calibration_nll=float(cal_base.mean()),
                native_zero_check="exact NLL equality", data=data_info, verdict=verdict,
                artifacts=["heads_calibration.csv", "metadata.json"],
                source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
            (output/"metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
            print(verdict)
            print(f"Saved calibration diagnostics in {output.resolve()}")
            return
        beta = st["xy"]/np.maximum(st["xx"], 1e-30)  # calibration-only signed fit
        grid = [0., -.25, .25, -.5, .5, -1., 1.]
        grid_loss = [float(losses(cal, a*gate_t).mean()) for a in grid]
        best = int(np.argmin(grid_loss))
        selected = grid[best] if grid_loss[0]-grid_loss[best] > 1e-7 else 0.
        # Check actual objective derivative against autograd; not an RoPE identity check.
        fd_checks = []
        if gate.any():
            for step in [.01, .005]:
                fd = (losses(cal, step*gate_t).mean()-losses(cal, -step*gate_t).mean())/(2*step)
                fd_checks.append(dict(step=step, finite_difference=float(fd), autograd=float(gradient[gate].sum())))
            error = min(abs(r["finite_difference"]-r["autograd"]) for r in fd_checks)
            if error > 2e-4+0.1*abs(float(gradient[gate].sum())):
                raise RuntimeError("Objective finite difference disagrees with intervention gradient.")
        print(f"Calibration complete: selected alpha={selected:+.2f}; evaluating actual interventions.", flush=True)
        test = load_blocks("test", args.test)
        baseline = losses(test)
        losses(test, zero, record=True)
        ts = aggregate()
        rows = []
        for h in range(H):
            yy, xx, xy = ts["yy"][h], ts["xx"][h], ts["xy"][h]
            rows.append(dict(layer=li, head=h, gradient=float(gradient[h]), chi=float(chi[h]),
                mean_lag=float(lag[h]), eligible=bool(gate[h]), strict_chi_eligible=bool(strict_gate[h]),
                valid_fraction=float(st["count"][h]/st["rows"]),
                clipped_fraction=float(st["clipped"][h]/st["rows"]),
                cal_signed_increment_scale=float(beta[h]),
                test_increment_r2_unit=float((2*xy-xx)/yy) if yy>0 else None,
                test_increment_r2_calibrated=float((2*beta[h]*xy-beta[h]**2*xx)/yy) if yy>0 else None,
                test_increment_signed_cosine=float(xy/math.sqrt(xx*yy)) if xx*yy>0 else None))
        write_csv(output/"heads.csv", rows)
        res = {"baseline": baseline, "main_alpha1": losses(test, gate_t),
               "sign_flip": losses(test, -gate_t),
               "calibrated": losses(test, selected*gate_t)}
        for mode in ["random", "permute"]:
            for seed in range(args.null_seeds):
                res[f"{mode}_{seed}"] = losses(test, gate_t, mode, args.seed+101+seed)
        write_csv(output/"per_article_losses.csv", [dict(article_block=i, scored_tokens=args.length-1,
            **{name: float(val[i]) for name, val in res.items()}) for i in range(len(test))])
        rng = np.random.default_rng(args.seed)
        contrasts = []
        comparisons = {"baseline": baseline, "sign_flip": res["sign_flip"],
                      **{mode: np.stack([res[f"{mode}_{s}"] for s in range(args.null_seeds)])
                         for mode in ["random", "permute"]}}
        for name, comparator in comparisons.items():
            contrasts.append(dict(comparator=name, **paired_ci(res["main_alpha1"], comparator, rng, args.bootstrap, args.contrast_family_size)))
        write_csv(output/"primary_contrasts.csv", contrasts)
        exploratory = paired_ci(res["calibrated"], baseline, rng, args.bootstrap)
        if args.smoke:
            verdict = "SOFTWARE SMOKE TEST ONLY; random weights provide no scientific evidence."
        elif all(r["ci_hi"] < 0 for r in contrasts):
            verdict = "EXPLORATORY EVIDENCE: fixed alpha=1 policy improves NLL versus baseline and specified controls. Independent replication required."
        else:
            verdict = "INCONCLUSIVE FOR FIXED ALPHA=1: benefit beyond all specified controls is not established; this does not refute the general hypothesis."
        versions = {}
        for pkg in ["torch", "transformers", "datasets", "numpy"]:
            try:
                versions[pkg] = importlib.metadata.version(pkg)
            except importlib.metadata.PackageNotFoundError:
                pass
        metadata = dict(protocol_version=3, arguments=vars(args), selected_layer=li, versions=versions,
            step_policy="uncapped" if args.uncapped else "relative_norm_cap",
            heldout_evaluated=True, active_heads=int(gate.sum()), strict_chi_heads=int(strict_gate.sum()),
            protocol_note="Version 3 adds calibration-only screening across all layers and an optional strict local-head gate. Screening workflow uses uncapped corrections and tests every qualifying layer. Previously inspected test articles are not independent confirmation.",
            model_commit=getattr(cfg, "_commit_hash", None), model_config=cfg.to_dict(),
            device=str(device), data=data_info, calibration_grid=grid, calibration_nll=grid_loss,
            selected_alpha=selected, objective_derivative_checks=fd_checks,
            native_zero_check="exact NLL equality", local_derivative_check="float64 central differences",
            calibrated_vs_baseline_exploratory=exploratory, verdict=verdict,
            primary_ci="Bonferroni-adjusted paired article/seed bootstrap; asymptotic, not a finite-sample guarantee",
            increment_metric="held-out uncentered R2 versus zero change; fixed-content yprime predictor",
            null_note="Permutation preserves causal key set; degenerate rows fall back to sign flip. Momentum permutes features.",
            source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
        (output/"metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
        print(verdict)
        print(f"Calibrated alpha={selected:+.2f}: test delta NLL={exploratory['delta_nll']:+.6g}, "
              f"exploratory 95% CI [{exploratory['ci_lo']:+.6g}, {exploratory['ci_hi']:+.6g}].")
        print(f"Saved heads.csv, per_article_losses.csv, primary_contrasts.csv, metadata.json in {output.resolve()}")
    finally:
        handle.remove()


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('Embedded: frozen_rope_gradient_test.py')


In [ ]:
#@title 2b. Embedded every-anchor controls (expand to inspect)
_ = (SOURCE_DIR / 'rope_baseline_replay.py').write_text(r'''#!/usr/bin/env python3
"""Measure fixed-prefix baselines and every frozen anchor from recorded factors.

Run: python rope_baseline_replay.py /content/lol --output /content/baselines
Requires NumPy and the original weights.npz, metadata.json, context_*.npz.
No model loading, new forward passes, fitting, head selection, or gauge changes.
Outputs describe prefix-only attention writes after Wo, not task performance.
"""
import argparse
import csv
import json
from pathlib import Path

import numpy as np


def mse(a, b):
    return float(np.mean(np.square(a - b)))


def ratio_root(a, b):
    return float(np.sqrt(a / b)) if b > 0 else None


def anchor_rows(context, head, native, frozen, mu, linear, token_ids):
    """frozen[a, i] applies anchor a's matrix to query i; exclude i == a."""
    rows = []
    for a in range(len(native)):
        keep = np.arange(len(native)) != a
        row = dict(context=context, head=head, anchor=a,
                   anchor_token_id=int(token_ids[a]), queries_tested=int(keep.sum()),
                   frozen_mse=mse(frozen[a, keep], native[keep]),
                   constant_anchor_mse=mse(native[a], native[keep]),
                   mu_only_mse=mse(mu, native[keep]),
                   linearized_softmax_mse=mse(linear[keep], native[keep]),
                   target_second_moment=float(np.mean(native[keep] ** 2)))
        row['frozen_relative_rmse'] = ratio_root(row['frozen_mse'], row['constant_anchor_mse'])
        row['frozen_vs_mu_relative_rmse'] = ratio_root(row['frozen_mse'], row['mu_only_mse'])
        rows.append(row)
    return rows


def evaluate_context(context_id, data, wo, head_to_kv):
    q, left, fast = [np.asarray(data[k], dtype=np.float64) for k in ['query_aug', 'L', 'T']]
    keys, values = [np.asarray(data[k], dtype=np.float64) for k in ['keys', 'values']]
    tokens = data['query_token_ids']
    h, queries, d, r = fast.shape
    if (queries < 2 or q.ndim != 2 or len(q) != queries or left.shape != (h, q.shape[1], d)
            or wo.shape[:2] != (h, r) or len(tokens) != queries or len(head_to_kv) != h):
        raise ValueError('Incompatible recorded factor shapes or fewer than two queries.')
    if not all(np.isfinite(x).all() for x in [q, left, fast, keys, values, wo]):
        raise ValueError('Nonfinite recorded arrays.')
    if keys.ndim != 3 or values.ndim != 3 or keys.shape[:2] != values.shape[:2] or keys.shape[1] == 0:
        raise ValueError('Expected nonempty shared [kv_heads, prefix_length, dimension] arrays.')
    if keys.shape[2] != d or values.shape[2] != r:
        raise ValueError('Key/value dimensions do not match the recorded factors.')
    np.testing.assert_allclose(q[:, -1], 1., atol=0, rtol=0)
    rows = []
    total_y = np.zeros((queries, wo.shape[-1]))
    total_f = np.zeros((queries, queries, wo.shape[-1]))
    total_mu = np.zeros(wo.shape[-1])
    total_lin = np.zeros_like(total_y)
    max_error = 0.
    for head in range(h):
        kv = int(head_to_kv[head])
        if not 0 <= kv < len(keys):
            raise ValueError('Invalid grouped-query head assignment.')
        read = q @ left[head]  # L already includes the query rotation and 1/sqrt(d).
        scores = read @ keys[kv].T
        p = np.exp(scores - scores.max(axis=1, keepdims=True))
        p /= p.sum(axis=1, keepdims=True)
        mu = values[kv].mean(axis=0)
        native = p @ values[kv]
        adaptive = mu + np.einsum('id,idr->ir', read, fast[head])
        np.testing.assert_allclose(adaptive, native, atol=3e-9, rtol=3e-8,
                                   err_msg='Recorded adaptive factors fail direct softmax reconstruction.')
        max_error = max(max_error, float(np.max(np.abs(adaptive - native))))
        frozen = mu + np.einsum('id,adr->air', read, fast[head])
        linear = mu + scores @ (values[kv] - mu) / len(values[kv])
        y, f, offset, lin = native @ wo[head], frozen @ wo[head], mu @ wo[head], linear @ wo[head]
        if not all(np.isfinite(x).all() for x in [y, f, offset, lin]):
            raise FloatingPointError('A projected readout exceeds the working numeric range.')
        rows.extend(anchor_rows(context_id, head, y, f, offset, lin, tokens))
        total_y += y
        total_f += f
        total_mu += offset
        total_lin += lin
    # Cross-head terms are preserved: sum projected writes before squaring.
    np.testing.assert_allclose(total_y, data['conditional_output'], atol=3e-9, rtol=3e-8)
    np.testing.assert_allclose(total_f[0], data['frozen_output'], atol=3e-9, rtol=3e-8)
    rows.extend(anchor_rows(context_id, 'all', total_y, total_f, total_mu, total_lin, tokens))
    return rows, max_error


def summarize(rows):
    result = []
    heads = sorted({r['head'] for r in rows}, key=lambda h: (h == 'all', 0 if h == 'all' else h))
    names = ['frozen_mse', 'constant_anchor_mse', 'mu_only_mse', 'linearized_softmax_mse',
             'target_second_moment']
    for head in heads:
        for setting in ['observed_anchor_0', 'all_anchors']:
            group = [r for r in rows if r['head'] == head and (setting == 'all_anchors' or r['anchor'] == 0)]
            contexts = sorted({r['context'] for r in group})
            # Equal context weight, then equal anchor weight within each context.
            out = dict(head=head, setting=setting, contexts=len(contexts), anchor_cases=len(group))
            for name in names:
                out[name] = float(np.mean([np.mean([r[name] for r in group if r['context'] == c]) for c in contexts]))
            out['target_rms'] = float(np.sqrt(out.pop('target_second_moment')))
            out['frozen_relative_rmse'] = ratio_root(out['frozen_mse'], out['constant_anchor_mse'])
            out['frozen_vs_mu_relative_rmse'] = ratio_root(out['frozen_mse'], out['mu_only_mse'])
            out['frozen_vs_linearized_relative_rmse'] = ratio_root(out['frozen_mse'], out['linearized_softmax_mse'])
            ratios = [r['frozen_relative_rmse'] for r in group if r['frozen_relative_rmse'] is not None]
            out['context_anchor_rrmse_min'] = min(ratios) if ratios else None
            out['context_anchor_rrmse_median'] = float(np.median(ratios)) if ratios else None
            out['context_anchor_rrmse_max'] = max(ratios) if ratios else None
            result.append(out)
    return result


def write_csv(path, rows):
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run(source, output):
    if source.resolve() == output.resolve():
        raise ValueError('Choose an output directory different from the source recordings.')
    metadata = json.loads((source / 'metadata.json').read_text())
    if metadata.get('matrix_convention') != 'section7a_original_scores_homogeneous':
        raise ValueError('Expected original-score fixed-prefix recordings, not exact_replay outputs.')
    files = sorted(source.glob('context_[0-9][0-9][0-9][0-9].npz'))
    if not files:
        raise FileNotFoundError('Missing context_*.npz; CSV summaries cannot recover the requested baselines.')
    with np.load(source / 'weights.npz', allow_pickle=False) as weights:
        wo = np.asarray(weights['Wo'], dtype=np.float64)
        mapping = np.asarray(weights['head_to_kv'])
    all_rows, checks = [], []
    for path in files:
        context = int(path.stem.split('_')[-1])
        with np.load(path, allow_pickle=False) as data:
            rows, error = evaluate_context(context, data, wo, mapping)
        all_rows.extend(rows)
        checks.append(dict(context=context, adaptive_max_abs_error=error))
        observed = next(r for r in rows if r['head'] == 'all' and r['anchor'] == 0)
        print(f"Context {context}: frozen={observed['frozen_mse']:.6g}; "
              f"mu={observed['mu_only_mse']:.6g}; linearized={observed['linearized_softmax_mse']:.6g}")
    output.mkdir(parents=True, exist_ok=True)
    write_csv(output / 'anchor_controls.csv', all_rows)
    summary = summarize(all_rows)
    write_csv(output / 'baseline_summary.csv', summary)
    info = dict(source=str(source.resolve()), model=metadata.get('model'), layer=metadata.get('layer'),
                checkpoint_kind=metadata.get('checkpoint_kind'), matrix_convention=metadata['matrix_convention'],
                attended_bank='Fixed prefix only; self K/V excluded.', new_model_forwards=0,
                pooling='Mean squared coordinate error over non-anchor queries; average anchors within context, then contexts.',
                relative_rmse='sqrt(pooled frozen MSE / pooled constant-anchor-output MSE)',
                linearized_softmax='mu + sum_j s_ij (v_j - mu) / N; no fitted scale.',
                checks=checks)
    (output / 'metadata.json').write_text(json.dumps(info, indent=2, allow_nan=False))
    for row in summary:
        if row['head'] == 'all':
            print(json.dumps(row, allow_nan=False))


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('source', type=Path)
    parser.add_argument('--output', type=Path)
    args = parser.parse_args()
    run(args.source, args.output or args.source / 'baseline_replay')
''', encoding='utf-8')
print('Embedded: rope_baseline_replay.py')


In [ ]:
#@title 2c. Embedded independent mathematical checks (expand to inspect)
_ = (SOURCE_DIR / 'release_checks.py').write_text(r'''"""Independent release checks; no checkpoint download or parameter training."""
import argparse
import json
from pathlib import Path

import numpy as np
import torch

import frozen_rope_gradient_test as engine
import rope_baseline_replay as controls


def run(output):
    engine.section7a_self_test()
    engine.section7a_refresh_self_test()
    rng = np.random.default_rng(20260905)
    errors = {name: 0.0 for name in [
        "gradient_step_max_abs", "position_derivative_max_abs",
        "softmax_position_derivative_max_abs", "all_anchor_oracle_max_abs"]}
    for trial in range(12):
        n, m, d, r, qn, heads = 9, 7, 4, 3, 6, 2
        query = np.column_stack((rng.normal(size=(qn, m)), np.ones(qn)))
        # A physical zero-input query checks the affine query-bias row.
        query[0, :-1] = 0
        wq = rng.normal(size=(heads, m + 1, d)) * .3
        keys = rng.normal(size=(heads, n, d)) * .5
        values = rng.normal(size=(heads, n, r))
        wo = rng.normal(size=(heads, r, m)) * .3
        freq = np.array([1., .03, 1., .03])
        position = 2.7
        cs, sn = np.cos(position * freq), np.sin(position * freq)
        matrices, lefts, fasts, ys, frozen, mus, linear = [], [], [], [], [], [], []
        for head in range(heads):
            result = engine.section7a_effective(
                query, wq[head], keys[head], values[head], cs, sn, d**-.5)
            slopes = result["left"] @ keys[head].T
            coeff = engine.section7a_coefficients(result["scores"])
            target = values[head] - result["mu"]
            assert np.all(coeff > 0)
            for qi in range(qn):
                B = torch.zeros((m + 1, r), dtype=torch.float64, requires_grad=True)
                a = torch.as_tensor(slopes.T, dtype=torch.float64)
                t = torch.as_tensor(target, dtype=torch.float64)
                c = torch.as_tensor(coeff[qi], dtype=torch.float64)
                # c, a, and t are fixed data when differentiating with respect to B.
                loss = .5 * torch.sum(c * torch.sum((a @ B - t)**2, dim=1))
                step = -torch.autograd.grad(loss, B)[0].detach().numpy()
                np.testing.assert_allclose(step, result["matrix"][qi], atol=2e-13, rtol=2e-12)
                errors["gradient_step_max_abs"] = max(errors["gradient_step_max_abs"],
                    float(np.max(np.abs(step - result["matrix"][qi]))))

            # Continuous read position; unrotated query content and the KV bank stay fixed.
            q = query[1] @ wq[head]
            def rotated(p):
                return engine.section7a_rotate(q, np.cos(p*freq), np.sin(p*freq))
            qrot = rotated(position)
            derivative = np.concatenate((-qrot[d//2:], qrot[:d//2])) * freq
            eps = 1e-5
            finite = (rotated(position+eps)-rotated(position-eps))/(2*eps)
            np.testing.assert_allclose(derivative, finite, atol=2e-9, rtol=2e-8)
            errors["position_derivative_max_abs"] = max(errors["position_derivative_max_abs"],
                float(np.max(np.abs(derivative-finite))))
            p = result["probability"][1]
            ds = derivative @ keys[head].T / np.sqrt(d)
            dy = (p * (ds - p @ ds)) @ values[head]
            def attention(pos):
                s = rotated(pos) @ keys[head].T / np.sqrt(d)
                prob = np.exp(s-s.max()); prob /= prob.sum()
                return prob @ values[head]
            fd = (attention(position+eps)-attention(position-eps))/(2*eps)
            np.testing.assert_allclose(dy, fd, atol=2e-9, rtol=2e-8)
            errors["softmax_position_derivative_max_abs"] = max(errors["softmax_position_derivative_max_abs"],
                float(np.max(np.abs(dy-fd))))

            matrices.append(result["matrix"])
            lefts.append(result["left"]); fasts.append(result["fast"])
            ys.append(result["native"] @ wo[head])
            frozen.append((result["mu"] + query @ result["matrix"][0]) @ wo[head])
            mus.append(result["mu"] @ wo[head])
            linear.append((result["mu"] + result["scores"] @ target / n) @ wo[head])
        data = dict(query_aug=query, L=np.stack(lefts), T=np.stack(fasts),
            keys=keys, values=values, query_token_ids=np.arange(qn),
            conditional_output=sum(ys), frozen_output=sum(frozen))
        rows, _ = controls.evaluate_context(trial, data, wo, np.arange(heads))
        # Dense, independently looped oracle: every anchor, every held-out query,
        # every head and the SUM of projected head writes before squaring.
        for anchor in range(qn):
            keep = np.arange(qn) != anchor
            for h in list(range(heads)) + ["all"]:
                chosen = range(heads) if h == "all" else [h]
                predicted = sum((values[k].mean(0) + query @ matrices[k][anchor]) @ wo[k]
                                for k in chosen)
                actual = sum(ys[k] for k in chosen)
                row = next(x for x in rows if x["head"] == h and x["anchor"] == anchor)
                references = dict(frozen_mse=float(np.mean((predicted[keep]-actual[keep])**2)),
                    constant_anchor_mse=float(np.mean((actual[anchor]-actual[keep])**2)),
                    mu_only_mse=float(np.mean((sum(mus[k] for k in chosen)-actual[keep])**2)),
                    linearized_softmax_mse=float(np.mean((sum(linear[k] for k in chosen)[keep]-actual[keep])**2)))
                for name, expected in references.items():
                    np.testing.assert_allclose(row[name], expected, atol=2e-13, rtol=2e-12)
                    errors["all_anchor_oracle_max_abs"] = max(errors["all_anchor_oracle_max_abs"], abs(row[name]-expected))
    record = dict(status="passed", fixture="synthetic, not pretrained", trials=12,
        gradient_steps_checked=12*heads*qn, errors=errors,
        checks=["independent relative-RoPE identity", "query/key/value affine biases",
            "zero and extreme scores", "GQA", "full self-KV replay", "corrupted-factor rejection",
            "autograd of the specified weighted least-squares objective at zero",
            "fixed-content positional derivative", "softmax positional derivative",
            "every frozen anchor and baseline against a dense oracle"])
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(record, indent=2, allow_nan=False) + "\n")
    print(json.dumps(record, indent=2, allow_nan=False))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", type=Path, required=True)
    run(parser.parse_args().output)
''', encoding='utf-8')
print('Embedded: release_checks.py')


In [ ]:
#@title 3. Prepare the isolated experiment environment
# Colab may omit ensurepip. Inherit its pip and CUDA PyTorch without calling ensurepip.
# Reconfigure on every run: a failed venv creation can leave bin/python present
# while pyvenv.cfg still disables access to Colab's installed packages.
# No --clear is used; existing environment packages and experiment results are preserved.
run_command([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', ENV_DIR],
            'create_environment', quiet=True)
PYTHON = ENV_DIR / 'bin/python'
pip_probe = subprocess.run([str(PYTHON), '-m', 'pip', '--version'], env=RUN_ENV,
                           stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if pip_probe.returncode:
    # Host pip can manage a pip-less environment directly; ensurepip remains unused.
    run_command([sys.executable, '-m', 'pip', '--python', PYTHON, 'install',
                 '--disable-pip-version-check', 'pip==25.1.1'], 'bootstrap_environment_pip', quiet=True)
requirements = [
    'transformers==4.51.3', 'datasets==3.5.0', 'numpy==2.1.3',
    'huggingface_hub==0.30.2', 'tokenizers==0.21.1', 'safetensors==0.5.3',
    'fsspec==2024.12.0', 'pandas==2.2.3',
]
(RUN_ROOT / 'requirements.txt').write_text('\n'.join(requirements)+'\n')
run_command([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', *requirements], 'install_dependencies', quiet=True)
torch_present = subprocess.run([str(PYTHON), '-c', 'import torch'], env=RUN_ENV, capture_output=True)
if torch_present.returncode:
    # A normal Colab runtime already has torch. This also makes smoke runs work on CPU-only Jupyter.
    run_command([PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url',
                 'https://download.pytorch.org/whl/cpu'], 'install_cpu_torch', quiet=True)
environment_code = "import importlib.metadata as m,json,torch,sys; print(json.dumps(dict(python=sys.version,versions={k:m.version(k) for k in ['torch','transformers','datasets','numpy','huggingface_hub','tokenizers','safetensors','fsspec','pandas']},cuda=torch.cuda.is_available(),device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')))"
environment = json.loads(run_command([PYTHON, '-c', environment_code], 'environment'))
(RUN_ROOT / 'environment.json').write_text(json.dumps(environment, indent=2)+'\n')
freeze = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], env=RUN_ENV, text=True)
(RUN_ROOT / 'environment_freeze.txt').write_text(freeze)
RUN_MANIFEST['environment'] = environment
RUN_MANIFEST['source_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(SOURCE_DIR.glob('*.py'))}
save_manifest()
if not environment['cuda'] and PROFILE != 'smoke':
    print('CPU execution selected. A Colab GPU runtime is recommended for the paper profile.')


In [ ]:
#@title 4. Verify the mathematics and every-anchor controls
run_command([PYTHON, SOURCE_DIR/'release_checks.py', '--output', RUN_ROOT/'mathematical_checks.json'], 'mathematical_checks')
math_checks = json.loads((RUN_ROOT/'mathematical_checks.json').read_text())
assert math_checks['status'] == 'passed'


## Pretrained measurement
Each article supplies one fixed prefix. Query 0 is its observed next token; the remaining queries are distinct, highest-ranked nonspecial continuations at the **same position**. Every query therefore reads the same prefix bank. The model's native full attention, including its own self key/value, is checked separately.

The command below computes each head's own $\Delta M_i=L T_i$, records all factors and token IDs, checks the full native head, and automatically replays both the literal identity and **anchor plus exact refresh**. A tiny random Qwen preflight runs before a pretrained checkpoint is downloaded. No head is removed by an eligibility gate. An unsuccessful numerical check stops release validation.


In [ ]:
#@title 5. Run --shared-update and automatic exact replay
# Rerunning this cell uses a new recording directory and preserves prior results.
if RESULTS.exists() and any(RESULTS.iterdir()):
    RESULTS = RUN_ROOT / ('recordings_' + datetime.now(timezone.utc).strftime('%H%M%S_%f'))
command = [PYTHON, SOURCE_DIR/'frozen_rope_gradient_test.py', '--shared-update',
           '--model', MODEL, '--revision', MODEL_REVISION, '--dataset-revision', DATASET_REVISION,
           '--out', RESULTS]
for name, value in CONFIG.items():
    command.extend(['--'+name, str(value)])
if PROFILE == 'smoke':
    command.append('--smoke')
RUN_MANIFEST.update(status='running', recordings=str(RESULTS.relative_to(RUN_ROOT)))
save_manifest()
run_command(command, 'pretrained_measurement' if PROFILE != 'smoke' else 'random_model_smoke')
measurement = json.loads((RESULTS/'metadata.json').read_text())
replay = json.loads((RESULTS/'exact_replay/metadata.json').read_text())
assert measurement['status'] == 'completed', measurement['status']
assert replay['status'] == 'completed', replay['status']
assert len(list(RESULTS.glob('context_*.npz'))) == CONFIG['contexts']
assert measurement['checkpoint_kind'] == ('random_smoke_model' if PROFILE == 'smoke' else 'pretrained')
if PROFILE != 'smoke':
    assert measurement['model_commit'] == MODEL_REVISION
print('PASS: every requested context was recorded and exact replay completed.')


In [ ]:
#@title 6. Compute μ-only, linearized-softmax, and every-anchor controls from the saved factors
BASELINES = RESULTS / 'baseline_replay'
run_command([PYTHON, SOURCE_DIR/'rope_baseline_replay.py', RESULTS, '--output', BASELINES], 'every_anchor_controls')


## Read the results
All output errors below concern attention writes **after $W_o$**. The `all` row sums the projected heads before squaring, retaining cross-head interactions. Fixed-prefix outputs and native full attention have separate reconstruction checks.

| Statistic | Definition |
|---|---|
| Exact MSE | Mean squared difference between the reconstructed and direct float64 softmax outputs. |
| Native FP32 check | Difference between float64 replay and captured native FP32 output; reported separately from algebraic exactness. |
| Frozen MSE | $\operatorname{mean}_{i\ne a,k}[(\mu W_o+u_i\Delta M_aW_o-y_iW_o)_k^2]$. |
| Relative RMSE | Square root of pooled frozen MSE divided by pooled constant-anchor-output MSE; $<1$ beats that constant output. |
| μ-only baseline | Broadcast the plain mean of the **current context's** values; it is query-independent. |
| Linearized-softmax baseline | $\mu+N^{-1}\sum_j s_{ij}(v_j-\mu)$, projected through $W_o$, with no fitted scale. This is only a baseline. |
| Matrix variation | $\sqrt{\sum_{c,i}\|\Delta M_{ci}W_o-\overline{\Delta MW_o}_c\|_F^2/\sum_{c,i}\|\Delta M_{ci}W_o\|_F^2}$, in the original-score gauge. |

MSE pooling gives equal weight to contexts and, in the every-anchor setting, equal weight to anchors within each context. Each anchor is excluded from its own transfer evaluation. Ratios use pooled errors, not averages of ratios; zero denominators remain undefined. `linear_update_query_variation` in the legacy CSV excludes the augmented bias row; it does **not** denote the linearized-softmax baseline.


In [ ]:
#@title 7. Show the combined-head result first, then every head
import pandas as pd
from IPython.display import display, Markdown

heads = pd.read_csv(RESULTS/'heads.csv', dtype={'head': str})
equivalence = pd.read_csv(RESULTS/'exact_replay/heads.csv', dtype={'head': str})
diagnostics = pd.read_csv(RESULTS/'diagnostics.csv', dtype={'head': str})
checks = pd.read_csv(RESULTS/'exact_replay/equivalence.csv', dtype={'head': str})
baselines = pd.read_csv(BASELINES/'baseline_summary.csv', dtype={'head': str})
anchor_cases = pd.read_csv(BASELINES/'anchor_controls.csv', dtype={'head': str})
expected = CONFIG['contexts'] * (measurement['heads'] + 1)
assert len(diagnostics) == expected and diagnostics['status'].eq('passed').all()
assert len(checks) == expected and checks['status'].eq('passed').all()
assert len(anchor_cases) == expected * CONFIG['queries']
assert heads['contexts_passed'].eq(CONFIG['contexts']).all()
assert baselines['contexts'].eq(CONFIG['contexts']).all()

display(Markdown(f"**{PROFILE.upper()} — {measurement['checkpoint_kind']}** · {CONFIG['contexts']} contexts · "
                 f"{CONFIG['queries']} queries · layer {measurement['layer']} · "
                 f"{expected}/{expected} context/head reconstruction rows passed."))
exact_columns = ['prefix_literal_mse', 'prefix_anchor_plus_refresh_mse', 'full_literal_mse',
                 'full_anchor_plus_refresh_mse', 'native_fp32_mse', 'native_fp32_max_abs_error']
# Fail explicitly if the report schema changes; do not silently omit a check.
missing = set(exact_columns) - set(equivalence.columns)
if missing:
    raise KeyError(f'Missing exact-replay columns: {sorted(missing)}')
display(Markdown('**Combined projected write: exact reconstruction and native precision**'))
display(equivalence.loc[equivalence['head'].eq('all'), ['head'] + exact_columns])
control_columns = ['head', 'setting', 'frozen_mse', 'constant_anchor_mse', 'mu_only_mse',
                   'linearized_softmax_mse', 'target_rms', 'frozen_relative_rmse']
display(Markdown('**Combined projected write: frozen transfer and controls**'))
display(baselines.loc[baselines['head'].eq('all'), control_columns])
per_head = heads[['head', 'contexts_passed', 'literal_update_query_variation']].merge(
    baselines.loc[baselines['setting'].eq('all_anchors'), control_columns], on='head', validate='one_to_one')
per_head = per_head.loc[~per_head['head'].eq('all')]
display(Markdown('**Every head: all anchors, original-score gauge**'))
display(per_head)
per_head.to_csv(RUN_ROOT/'per_head_release_summary.csv', index=False)
RUN_MANIFEST.update(status='verified', contexts_passed=CONFIG['contexts'],
                    reconstruction_rows_passed=expected, every_anchor_cases=len(anchor_cases))
save_manifest()


## Download and replay
The bundle includes `weights.npz`, every `context_*.npz`, exact replay outputs, every-anchor baselines, all CSVs, the report, source files, logs, and the run manifest. Dense matrices are recoverable without approximation as `DeltaM[h,i] = L[h] @ T[h,i]`; storing these exact factors avoids duplicating large arrays.

After extracting the ZIP, both replay commands in `README.md` run with **NumPy only**, without another model forward pass or download. `mathematical_checks.json` describes synthetic checks; the checkpoint metadata identifies actual pretrained measurements. The ZIP excludes downloaded model files and the virtual environment.


In [ ]:
#@title 8. Export the complete reproducibility bundle
assert RUN_MANIFEST['status'] in {'verified', 'completed'}, 'A complete successful run is required before release export.'
relative_recordings = RESULTS.relative_to(RUN_ROOT).as_posix()
readme = f"""# RoPE–softmax exact effective weights: reproducibility bundle

Profile: {PROFILE}; checkpoint kind: {measurement['checkpoint_kind']}.
Model: {MODEL}; commit: {MODEL_REVISION}.
Dataset commit: {DATASET_REVISION}.
Configuration: {json.dumps(CONFIG, sort_keys=True)}.
All {RUN_MANIFEST['reconstruction_rows_passed']} requested context/head rows passed.

Fresh Colab run: open the distributed notebook, select a GPU runtime, and Run all.
This bundle was generated by that notebook; results are never prepopulated.

Replay from this directory (NumPy only; choose new output directories):
  python -m pip install numpy==2.1.3
  python source/frozen_rope_gradient_test.py --replay {relative_recordings} --out replay_again
  python source/rope_baseline_replay.py {relative_recordings} --output baselines_again

Primary files:
- {relative_recordings}/heads.csv: pooled matrix variation and observed-anchor transfer.
- {relative_recordings}/diagnostics.csv: every context/head, including all-head projected sums.
- {relative_recordings}/exact_replay/heads.csv: exact prefix, full attention, refresh, and FP32 checks.
- {relative_recordings}/baseline_replay/baseline_summary.csv: mu-only, linearized-softmax, all anchors.
- {relative_recordings}/baseline_replay/anchor_controls.csv: each individual anchor case.
- {relative_recordings}/weights.npz and context_*.npz: all exact replay inputs.
- mathematical_checks.json: synthetic identity, derivative, autograd and control-oracle checks.
- environment.json, environment_freeze.txt, requirements.txt, run_manifest.json: reproducibility.
- SHA256SUMS.txt: checksums for all other bundled files.

The original-score gauge is fixed. Mu is the plain mean of the current attended values.
Matrix inputs are after RMSNorm, augmented by 1. Prefix transfer excludes self K/V;
full native attention is checked separately. All-head outputs are summed before squaring.
Relative RMSE = sqrt(pooled frozen MSE / pooled constant-anchor-output MSE).
Every-anchor statistics exclude the anchor query, average anchors within a context,
then average contexts. A frozen control omits u_i(DeltaM_i-DeltaM_anchor);
the exact refresh adds precisely that term. Linearized softmax is only a comparator.

Sources:
- https://huggingface.co/Qwen/Qwen2.5-0.5B/tree/{MODEL_REVISION}
- https://huggingface.co/datasets/Salesforce/wikitext/tree/{DATASET_REVISION}
- https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/qwen2
"""
(RUN_ROOT/'README.md').write_text(readme, encoding='utf-8')
RUN_MANIFEST.update(status='completed', completed_utc=datetime.now(timezone.utc).isoformat())
save_manifest()
root_files = {'README.md', 'requirements.txt', 'environment.json', 'environment_freeze.txt',
              'run_manifest.json', 'mathematical_checks.json', 'per_head_release_summary.csv'}
def intended_artifact(p):
    relative = p.relative_to(RUN_ROOT)
    parts = relative.parts
    if len(parts) == 1:
        return p.name in root_files
    if parts[0] == 'source':
        return len(parts) == 2 and p.suffix == '.py'
    if parts[0] == 'logs':
        return len(parts) == 2 and p.suffix == '.log'
    return parts[0] == RESULTS.name and p.suffix in {'.npz', '.csv', '.json', '.md'}
files_to_ship = sorted(p for p in RUN_ROOT.rglob('*') if p.is_file() and intended_artifact(p))
checksums = '\n'.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(RUN_ROOT).as_posix()}'
                      for p in files_to_ship) + '\n'
(RUN_ROOT/'SHA256SUMS.txt').write_text(checksums)
files_to_ship.append(RUN_ROOT/'SHA256SUMS.txt')
ARCHIVE = RUN_ROOT.with_suffix('.zip')
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for p in files_to_ship:
        archive.write(p, arcname=p.relative_to(RUN_ROOT))
with zipfile.ZipFile(ARCHIVE) as archive:
    assert archive.testzip() is None
print(f'COMPLETE: {len(files_to_ship)} files, {ARCHIVE.stat().st_size / 2**20:.1f} MiB')
print(ARCHIVE)
try:
    from google.colab import files
except ImportError:
    print('Outside Colab: the ZIP is saved at the path above.')
else:
    if AUTO_DOWNLOAD:
        files.download(str(ARCHIVE))
    else:
        print('Automatic download disabled; download the ZIP from the Colab Files panel.')


Implementation references: [Qwen2 documentation, Transformers 4.51.3](https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/qwen2), [pinned Qwen2.5 checkpoint](https://huggingface.co/Qwen/Qwen2.5-0.5B/tree/060db6499f32faf8b98477b0a26969ef7d8b9987), and [pinned WikiText dataset](https://huggingface.co/datasets/Salesforce/wikitext/tree/b08601e04326c79dfdd32d625aee71d232d685c3).
